# VitaCall, ML Engineering & Ops

Auteur: Thomas en Parsa

VitaCall is een Nederlandse alarmcentrale. In dit notebook train ik twee sentiment-modellen voor binnenkomende gesprekken. Het zware model draait achter een FastAPI service. Het lichte model zit in een PySide6 desktop-app en werkt offline.

Alle code staat hieronder. Geen aparte `backend/`-map. Modellen gaan naar `models/`, dataset-stappen naar `data/`, en de JSON voor de desktop-app naar `models/sentiment_lite.json`.

## Leerdoelen, wat in welk hoofdstuk

| LD | Onderwerp | Sectie | Hoe |
|----|-----------|--------|-----|
| 1  | Datapipeline (ETL + validatie)  | 1    | drie lagen: ruw, schoon, klaar voor training, met checks per laag |
| 2  | Schaalbaarheid                  | 1.4  | PySpark als de JVM beschikbaar is, anders pandas. Streaming per batch. |
| 3  | Modellering & ML-pipeline       | 2    | sklearn pipeline, MLflow, hyperparam-sweep, federated learning |
| 4  | Deployment                      | 3    | FastAPI, Docker, desktop, GitHub Actions |
| 5  | Monitoring                      | 4    | DriftDetector, Metrics p50/p95, /metrics endpoint, JSON-logs |

## Hoe lees je dit

Elke sectie begint met een korte uitleg, daarna de code. De cellen draaien in volgorde van boven naar beneden. Onder elke codecel zie je de uitvoer, of een foutmelding als iets misgaat. Die laatste is ook leerzaam: dan zie je waar de pijplijn vastloopt.

## 0. Productvereisten (rubric: stakeholders + model-eisen + retraining)

### Stakeholders
| Rol | Belang |
|---|---|
| Operator alarmcentrale | krijgt realtime sentiment + spoed-keywords in UI |
| Beller | krijgt snellere triage zonder dat audio het pand verlaat (privacy) |
| MLOps engineer | beheert pipeline, modellen, CI/CD/CT |
| Compliance officer | bewaakt AVG-conformiteit (audio blijft on-device bij edge-pad) |
| Productmanager | stuurt op SLA's (uptime, p95-latency, drift-alerts) |

### Model-eisen (SLA)
| Eis | Doel | Werkelijk |
|---|---|---|
| Test-accuracy heavy | >= 0.85 | 0.871 (`evidence/cv_scores.json`) |
| CV-F1 mean (5-fold) | >= 0.80 | 0.848 ± 0.056 |
| Pickle-grootte heavy | <= 1 MB | 0.22 MB |
| Pickle-grootte lite | <= 100 KB | 36 KB |
| Inference-latency (p95) | <= 50 ms | gemeten in `/metrics`, ~5-15 ms |
| Endpoints | `/health`, `/analyze`, `/drift`, `/metrics` | alle vier live in `serve.py` |

### Retraining-strategie
1. **Trigger 1 — schedule**: `cicd.yml` (job: retrain) draait elke zondag 03:00 UTC.
2. **Trigger 2 — drift**: als `drift_score > 0.30` (output) of PSI > 0.20 (input), alert-engine zet `retrain_recommended=true` in `evidence/alerts.jsonl`.
3. **Trigger 3 — handmatig**: `workflow_dispatch` in GitHub UI.
4. **Validatie-gate**: nieuwe model wordt alleen gepromoveerd als CV-F1 >= huidige - 0.01 (anti-regression). Anders artifact-only.

### Data-eisen (samengevat)
- **Kwaliteit**: 3-laags validatie (`validate_*`), fail-fast bij errors.
- **Volume**: streaming via `pyarrow.iter_batches` (constant geheugen).
- **Snelheid**: PySpark voor parallelle split, fsspec voor cloud-storage.
- **Privacy**: edge-model scoort lokaal in `app/`; cloud-pad alleen voor tekstuele transcripten, geen audio.
- **Veiligheid**: model-bestanden gepind via SHA256 in `data/MANIFEST.json`.


In [1]:
# Eenmalig: dependencies installeren als ze nog niet in je environment zitten.

In [2]:
# Standaard imports voor het hele notebook. We bundelen ze hier zodat
import glob
import json
import logging
import os
import pickle
import tarfile
import tempfile
import time
from collections import deque
from contextlib import contextmanager
from dataclasses import dataclass, field
from functools import reduce

import numpy as np
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.pipeline import Pipeline

# Versie-check voor reproduceerbaarheid: een crashende notebook half jaar
import sklearn
print(f'Python pakketten: pandas={pd.__version__}, numpy={np.__version__}, '
      f'sklearn={sklearn.__version__}')

# Pad-constanten. Het notebook draait vanuit de project-root; de pipeline-data
DATA_DIR = 'data'
MODEL_DIR = 'models'
EDGE_DIR = 'models'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Reproduceerbaarheid: één seed door het hele notebook. Spark en
# numpy-permutaties gebruiken deze ook.
RANDOM_SEED = 42

# Eenvoudige logger zodat je in de output ziet wat de pipeline doet.
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger('vitacall')

Python pakketten: pandas=2.3.3, numpy=2.3.4, sklearn=1.7.2


## 1. Datapipeline (LD1 + LD2)

De data gaat in drie stappen door de pijp. Per stap valideren we het schema. Fouten in de eerste laag worden alleen maar erger als je ze niet vroeg vangt.

1. Ruwe laag: DBRD wordt gedownload, de tekst-bestanden worden samengevoegd tot een tabel.
2. Schone laag: HTML eruit, witruimte normaliseren, duplicaten weg.
3. Trainings-laag: stratified 80/10/10 split per label, plus `token_count` als feature.

Elke `validate_*`-functie geeft een `ValueError` met de exacte reden als een check faalt. Geen stille corruptie verderop.

### 1.1 Validatie

Een dataclass `ValidationResult` houdt errors en warnings bij. De `validate_*`-functies vullen die. Bewust geen Pandera of Great Expectations: de checks zijn klein genoeg om met de hand te schrijven, en zo zie je in een oogopslag wat er gecontroleerd wordt.

In [3]:
# Container voor het resultaat van een validatie. We verzamelen alle problemen
# in één pass; pas daarna besluiten we of de pipeline mag doorgaan.
@dataclass
class ValidationResult:
    layer: str
    n_rows: int
    n_cols: int
    errors: list = field(default_factory=list)
    warnings: list = field(default_factory=list)

    @property
    def passed(self) -> bool:
        return not self.errors

    def raise_if_failed(self) -> 'ValidationResult':
        # Fail-fast: als er een echte fout is, stoppen we de pipeline meteen.
        if not self.passed:
            raise ValueError(f'Validatie {self.layer} faalde: {self.errors}')
        return self


def _check(result, cond, msg, severity='error'):
    # Helper: voeg een melding toe als de conditie False is.
    if not cond:
        (result.errors if severity == 'error' else result.warnings).append(msg)


def validate_raw(df):
    # Eerste laag: alle vereiste kolommen aanwezig, label is 0 of 1, niet leeg.
    r = ValidationResult('ruw', len(df), len(df.columns))
    for col in ('review_id', 'text', 'label', 'source_file'):
        _check(r, col in df.columns, f'kolom {col!r} ontbreekt')
    if 'label' in df.columns:
        _check(r, df['label'].isin([0, 1]).all(), 'label moet 0 of 1 zijn')
    _check(r, len(df) > 0, 'tabel is leeg')
    return r


def validate_clean(df):
    # Tweede laag: text_clean bestaat, niets is leeg, geen HTML-tags.
    r = ValidationResult('schoon', len(df), len(df.columns))
    for col in ('text_clean', 'label', 'split'):
        _check(r, col in df.columns, f'kolom {col!r} ontbreekt')
    if 'text_clean' in df.columns:
        _check(r, df['text_clean'].notna().all(), 'text_clean bevat NaN')
        _check(r, (df['text_clean'].str.len() > 0).all(), 'text_clean bevat lege strings')
        _check(r, not df['text_clean'].str.contains(r'<[a-zA-Z/]', regex=True).any(),
               'HTML-tags niet verwijderd')
    return r


def validate_train_ready(df):
    # Derde laag: 80/10/10 split aanwezig, token_count berekend.
    r = ValidationResult('trainklaar', len(df), len(df.columns))
    for col in ('text_clean', 'label', 'split', 'token_count'):
        _check(r, col in df.columns, f'kolom {col!r} ontbreekt')
    if 'split' in df.columns:
        splits = set(df['split'].unique())
        _check(r, splits == {'train', 'val', 'test'},
               f'verwacht train/val/test, kreeg {splits}')
    return r


def report(result):
    # Mens-leesbare samenvatting voor in de notebook-output.
    icon = 'OK' if result.passed else 'FAIL'
    lines = [f'[{icon}] {result.layer}: {result.n_rows:,} rijen x {result.n_cols} kolommen']
    for e in result.errors:
        lines.append(f'    ERROR: {e}')
    for w in result.warnings:
        lines.append(f'    WARN:  {w}')
    return '\n'.join(lines)


# Snelle sanity-check: een mini-frame valideren zonder echte data.
_demo = pd.DataFrame({'text_clean': ['oke'], 'label': [1], 'split': ['train']})
print(report(validate_clean(_demo)))

[OK] schoon: 1 rijen x 3 kolommen


### 1.2 Ruwe laag, Nederlandse boekenrecensies downloaden en samenvoegen

We gebruiken de Dutch Book Reviews Dataset (DBRD): het labeled deel van DBRD (~22k recensies; 110k incl. unsup) van Hebban.nl, met sentiment-labels (positief/negatief), uitgegeven door Benjamin van der Burgh (Universiteit Leiden, 2019). Veel relevanter voor VitaCall dan een Engelstalige IMDb-set: het model leert direct Nederlandse zinsbouw en woordgebruik.

De tar.gz wordt een keer gedownload en uitgepakt. Daarna leest `ingest_dbrd` de losse `.txt`-bestanden uit `train/{pos,neg}` en `test/{pos,neg}` in en schrijft alles als een Parquet-tabel weg. Idempotent: als de download er al staat, slaan we die over.

In [4]:
# Bron: https://github.com/benjaminvdb/110kDBRD - 110k Nederlandse boekenrecensies van Hebban.nl
DBRD_URL = 'https://github.com/benjaminvdb/110kDBRD/releases/download/v3.0/DBRD_v3.tgz'
# Download-foutmelding netjes opvangen.
try:
    requests.head(DBRD_URL, timeout=5)
except requests.RequestException as e:
    log.warning('DBRD-URL niet bereikbaar: %s, fallback gebruikt als beschikbaar', e)



def download_dbrd(base_dir):
    # Download de DBRD tar.gz naar base_dir/DBRD/. Idempotent: skipt als het er al staat.
    out_dir = os.path.join(base_dir, 'DBRD')
    if os.path.exists(out_dir) and any(os.scandir(out_dir)):
        log.info('DBRD al uitgepakt in %s', out_dir)
        return out_dir
    tar_path = os.path.join(base_dir, 'DBRD_v3.tgz')
    if not os.path.exists(tar_path):
        log.info('DBRD downloaden (eenmalig, ~155 MB)...')
        with requests.get(DBRD_URL, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(tar_path, 'wb') as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
    log.info('DBRD uitpakken...')
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(path=base_dir)
    return out_dir


def ingest_dbrd(dbrd_dir, out):
    # Ruwe laag: lees alle .txt-recensies uit train/{pos,neg} en test/{pos,neg}
    # en schrijf ze als een Parquet met review_id, text, label, source_file.
    rows = []
    for split in ['train', 'test']:
        for sentiment, label in [('pos', 1), ('neg', 0)]:
            sdir = os.path.join(dbrd_dir, split, sentiment)
            if not os.path.isdir(sdir):
                # Niet alle dataset-versies hebben dezelfde layout; we slaan stil over.
                continue
            for fname in os.listdir(sdir):
                if not fname.endswith('.txt'):
                    continue
                with open(os.path.join(sdir, fname), encoding='utf-8') as f:
                    rows.append((
                        f'{split}_{sentiment}_{fname[:-4]}',
                        f.read(),
                        label,
                        f'{split}/{sentiment}/{fname}',
                    ))
    if not rows:
        raise RuntimeError(f'Geen recensies gevonden in {dbrd_dir} - is de dataset goed uitgepakt?')
    os.makedirs(out, exist_ok=True)
    df = pd.DataFrame(rows, columns=['review_id', 'text', 'label', 'source_file'])
    df.to_parquet(os.path.join(out, 'reviews.parquet'), index=False)
    log.info('Ruwe laag: %d recensies geschreven naar %s', len(df), out)


# Uitvoeren, als de download faalt vangen we het netjes en tonen we de melding
# zodat het notebook niet halverwege onleesbaar crasht.
raw_dir = os.path.join(DATA_DIR, 'ruw', 'dbrd')
try:
    dbrd_dir = download_dbrd(DATA_DIR)
    ingest_dbrd(dbrd_dir, raw_dir)
    df_raw = pd.read_parquet(os.path.join(raw_dir, 'reviews.parquet'))
    print(report(validate_raw(df_raw).raise_if_failed()))
    df_raw.head(3)
except Exception as e:
    print(f'[FOUT] Ruwe laag faalde: {type(e).__name__}: {e}')
    print('       Check je internet of de DBRD-URL. Pipeline kan niet door zonder data.')
    raise

2026-05-13 18:59:40,914 [INFO] DBRD al uitgepakt in data\DBRD


2026-05-13 18:59:45,206 [INFO] Ruwe laag: 22252 recensies geschreven naar data\ruw\dbrd


[OK] ruw: 22,252 rijen x 4 kolommen


### 1.3 Schone laag, cleaning + validatie

Recensies bevatten HTML-resten, dubbele spaties en duplicaten. Die strippen we eruit voor we naar features gaan. De cleaning is een aparte functie (`_clean_df`) zodat de streaming-variant in 1.4 dezelfde transformaties hergebruikt zonder copy-paste.

In [5]:
def _clean_df(df):
    # Gedeelde cleaning: HTML-tags eruit, witruimte normaliseren, train/test
    df = df.copy()
    df['text_clean'] = (df['text']
                        .str.replace(r'<[^>]+>', ' ', regex=True)
                        .str.replace(r'\s+', ' ', regex=True)
                        .str.strip())
    df['split'] = df['source_file'].str.split('/').str[0]
    return df[df['label'].isin([0, 1])
              & df['text_clean'].notna()
              & (df['text_clean'] != '')]


def clean_reviews(raw_path, out):
    # Schone laag: HTML strippen, whitespace normaliseren, duplicaten weg.
    df = (_clean_df(pd.read_parquet(raw_path))
          .drop_duplicates(subset=['text_clean'])
          .reset_index(drop=True))
    os.makedirs(out, exist_ok=True)
    df.to_parquet(os.path.join(out, 'reviews.parquet'), index=False)
    log.info('Schone laag: %d unieke recensies (van %d ruw)', len(df), len(pd.read_parquet(raw_path)))


# Uitvoeren met try/except zodat een fout zichtbaar wordt zonder notebook-stack.
clean_dir = os.path.join(DATA_DIR, 'schoon', 'dbrd')
try:
    clean_reviews(os.path.join(raw_dir, 'reviews.parquet'), clean_dir)
    df_clean = pd.read_parquet(clean_dir)
    print(report(validate_clean(df_clean).raise_if_failed()))
    df_clean[['text_clean', 'label', 'split']].head(3)
except ValueError as e:
    print(f'[FOUT] Validatie schone laag: {e}')
    raise
except Exception as e:
    print(f'[FOUT] Cleaning faalde: {type(e).__name__}: {e}')
    raise

2026-05-13 18:59:48,675 [INFO] Schone laag: 22249 unieke recensies (van 22252 ruw)


[OK] schoon: 22,249 rijen x 6 kolommen


### 1.4 Schaalbaarheid (LD2), streaming voor data > RAM

22k recensies past ruim in geheugen. Als VitaCall straks honderdduizenden gesprekken per maand verwerkt is dat niet meer vanzelfsprekend. `clean_reviews_streaming` leest de Parquet in batches van 5000 rijen via `pyarrow.parquet.ParquetFile.iter_batches`. Dezelfde transformaties, constante geheugenfootprint.

In productie zou dit een Spark-job of een Airflow-DAG worden. Voor dit notebook is de streaming-variant genoeg om aan te tonen dat de pipeline niet stuk gaat op grote datasets.

In [6]:
def clean_reviews_streaming(raw_path, out, chunk_size=5000):
    # Streaming-variant: leest de Parquet in batches en past dezelfde
    import pyarrow as pa
    import pyarrow.parquet as pq

    pf = pq.ParquetFile(raw_path)
    os.makedirs(out, exist_ok=True)
    n_total = 0
    writer = None
    try:
        for batch in pf.iter_batches(batch_size=chunk_size):
            df = _clean_df(batch.to_pandas())
            n_total += len(df)
            table = pa.Table.from_pandas(df, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(os.path.join(out, 'reviews.parquet'), table.schema)
            writer.write_table(table)
    finally:
        if writer:
            writer.close()
    return n_total


# Demo: streaming-variant op de ruwe Parquet, output in een tijdelijke map.
try:
    with tempfile.TemporaryDirectory() as tmp:
        n = clean_reviews_streaming(os.path.join(raw_dir, 'reviews.parquet'), tmp, chunk_size=5000)
        out_size = os.path.getsize(os.path.join(tmp, 'reviews.parquet')) / 1024 / 1024
        print(f'Streaming verwerkte {n:,} rijen in chunks van 5000')
        print(f'Output Parquet: {out_size:.1f} MB')
except FileNotFoundError as e:
    print(f'[FOUT] Geen ruwe Parquet gevonden: {e}')
    print('       Sectie 1.2 moet eerst gedraaid hebben.')
    raise

Streaming verwerkte 22,252 rijen in chunks van 5000
Output Parquet: 38.7 MB


### 1.5 Trainings-laag, stratified split (PySpark, met pandas-fallback)

PySpark verdeelt het werk over alle CPU-cores via `master("local[*]")`. Op Windows zonder `winutils.exe` faalt de Spark-init. In dat geval gaat de pijplijn over op een pandas-equivalent met dezelfde stratified 80/10/10 split. Stratified per label voorkomt class-imbalance per split. Anders krijg je in het slechtste geval een test-set met 90% positief.

In [7]:
def get_spark(app='VitaCall'):
    # Lokale Spark-sessie. spark.ui.enabled=false omdat we geen webserver
    from pyspark.sql import SparkSession
    return (SparkSession.builder.master('local[*]').appName(app)
            .config('spark.sql.shuffle.partitions', '4')
            .config('spark.ui.enabled', 'false')
            .config('spark.driver.memory', '4g')
            .getOrCreate())


def features_with_spark(spark, clean_dir, out, seed=42):
    # PySpark-pad: per label een 80/10/10 randomSplit, daarna unionByName
    # en token_count erbij. randomSplit met dezelfde seed = reproduceerbaar.
    from pyspark.sql import functions as F
    df = spark.read.parquet(clean_dir)
    parts = []
    for label_val in [0, 1]:
        tr, va, te = df.filter(F.col('label') == label_val).randomSplit([0.8, 0.1, 0.1], seed=seed)
        parts += [tr.withColumn('split', F.lit('train')),
                  va.withColumn('split', F.lit('val')),
                  te.withColumn('split', F.lit('test'))]
    (reduce(lambda a, b: a.unionByName(b), parts)
     .withColumn('token_count', F.size(F.split(F.col('text_clean'), r'\s+')))
     .write.mode('overwrite').parquet(out))


def features_with_pandas(clean_dir, out, seed=42):
    # Fallback zonder JVM: zelfde stratified split met numpy permutations.
    df = pd.read_parquet(clean_dir)
    rng = np.random.default_rng(seed)
    parts = []
    for label_val in [0, 1]:
        sub = df[df['label'] == label_val].copy()
        sub = sub.iloc[rng.permutation(len(sub))]
        n = len(sub)
        n_tr, n_va = int(n * 0.8), int(n * 0.1)
        sub['split'] = ['train'] * n_tr + ['val'] * n_va + ['test'] * (n - n_tr - n_va)
        parts.append(sub)
    out_df = pd.concat(parts, ignore_index=True)
    out_df['token_count'] = out_df['text_clean'].str.split().str.len()
    os.makedirs(out, exist_ok=True)
    out_df.to_parquet(os.path.join(out, 'reviews.parquet'), index=False)


# Probeer eerst Spark; als de JVM niet start (typisch op Windows zonder
# winutils), val terug op pandas. We rapporteren welke engine gebruikt is.
train_dir = os.path.join(DATA_DIR, 'trainklaar', 'dbrd')
engine = None
try:
    spark = get_spark()
    spark.sparkContext.setLogLevel('ERROR')
    features_with_spark(spark, clean_dir, train_dir)
    spark.stop()
    engine = 'PySpark'
except Exception as e:
    log.warning('PySpark niet beschikbaar (%s); fallback naar pandas', type(e).__name__)
    features_with_pandas(clean_dir, train_dir)
    engine = f'pandas (fallback: {type(e).__name__})'

df_train_ready = pd.read_parquet(train_dir)
print(f'Engine: {engine}')
print(report(validate_train_ready(df_train_ready).raise_if_failed()))
print(df_train_ready.groupby(['split', 'label']).size())

2026-05-13 19:00:03,454 [WARNING] PySpark niet beschikbaar (Py4JJavaError); fallback naar pandas


Engine: pandas (fallback: Py4JJavaError)
[OK] trainklaar: 22,249 rijen x 7 kolommen
split  label
test   0        1113
       1        1113
train  0        8899
       1        8900
val    0        1112
       1        1112
dtype: int64


### 1.6 Reproduceerbaarheid: dataset-manifest met SHA256 + rij-counts

Voor versiebeheer van data schrijven we `data/MANIFEST.json` met per laag het pad, aantal rijen, kolommen, label-verdeling en SHA256-checksum van de Parquet. Plus de git-SHA en de Python-versie. Zo kan iedereen achteraf reproduceren welke dataset bij welk model hoorde, zonder de dataset zelf te committen.

In [8]:
import hashlib
import platform
import subprocess
import sys


def _sha256(path):
    # Stream de file in 1MB blokken zodat ook grote Parquet-files passen.
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for blk in iter(lambda: f.read(1 << 20), b''):
            h.update(blk)
    return h.hexdigest()


def _git_sha():
    try:
        out = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'],
                                      stderr=subprocess.DEVNULL).decode().strip()
        return out
    except Exception:
        return 'unknown'


def build_manifest():
    layers = [
        ('ruw',         os.path.join(raw_dir,   'reviews.parquet')),
        ('schoon',      os.path.join(clean_dir, 'reviews.parquet')),
        ('trainklaar',  os.path.join(train_dir, 'reviews.parquet')),
    ]
    entries = []
    for layer, path in layers:
        if not os.path.exists(path):
            entries.append({'layer': layer, 'path': path, 'status': 'missing'})
            continue
        df = pd.read_parquet(path)
        entries.append({
            'layer':     layer,
            'path':      path,
            'rows':      int(len(df)),
            'cols':      list(df.columns),
            'sha256':    _sha256(path),
            'size_mb':   round(os.path.getsize(path) / 1024 / 1024, 2),
            'label_dist': (df['label'].value_counts().to_dict()
                           if 'label' in df.columns else None),
        })
    return {
        'generated_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
        'git_sha':      _git_sha(),
        'python':       sys.version.split()[0],
        'platform':     platform.platform(),
        'seed':         RANDOM_SEED,
        'sklearn':      sklearn.__version__,
        'pandas':       pd.__version__,
        'numpy':        np.__version__,
        'layers':       entries,
    }


manifest = build_manifest()
manifest_path = os.path.join(DATA_DIR, 'MANIFEST.json')
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f'Manifest geschreven: {manifest_path}')
print(f'  git_sha={manifest["git_sha"]}  python={manifest["python"]}  seed={manifest["seed"]}')
for e in manifest['layers']:
    if e.get('rows') is not None:
        print(f'  [{e["layer"]:>10s}] rows={e["rows"]:>7,}  sha256={e["sha256"][:12]}...  size={e["size_mb"]} MB')
    else:
        print(f'  [{e["layer"]:>10s}] {e["status"]}')

Manifest geschreven: data\MANIFEST.json
  git_sha=b23e016  python=3.13.9  seed=42
  [       ruw] rows= 22,252  sha256=cb0ab5d9c29c...  size=19.51 MB
  [    schoon] rows= 22,249  sha256=c1721d384ae5...  size=38.62 MB
  [trainklaar] rows= 22,249  sha256=b32fb34de681...  size=38.73 MB


In [9]:
# LD1 evidence: persisteer alle drie de validatie-resultaten als JSON
# zodat een reviewer / CI-pipeline de pipeline-kwaliteit kan auditen.
validation_report = {
    'generated_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'git_sha': manifest['git_sha'],
    'layers': [],
}
for name, df, fn in [
    ('ruw',        df_raw,         validate_raw),
    ('schoon',     df_clean,       validate_clean),
    ('trainklaar', df_train_ready, validate_train_ready),
]:
    res = fn(df)
    validation_report['layers'].append({
        'layer': res.layer, 'passed': res.passed,
        'rows': res.n_rows, 'cols': res.n_cols,
        'errors': res.errors, 'warnings': res.warnings,
    })
os.makedirs('evidence', exist_ok=True)
with open('evidence/validation_report.json', 'w', encoding='utf-8') as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)
print('evidence/validation_report.json:',
      sum(l['passed'] for l in validation_report['layers']), '/ 3 lagen geslaagd')

evidence/validation_report.json: 3 / 3 lagen geslaagd


### 1.7 Schaalbaarheid: throughput benchmark + cloud-storage abstractie

We meten hoe snel de cleaning loopt op verschillende batch-sizes. Verder laten we zien dat de pipeline cloud-ready is: het pad-prefix kan zonder code-wijziging naar `s3://bucket/...` of `gs://bucket/...` via fsspec. We schrijven hier alleen de adapter-laag, geen echte cloud-credentials nodig om het patroon te tonen.

In [10]:
# Benchmark: streaming cleaning op verschillende batch-sizes.
# We meten throughput (rijen/sec) en peak memory (psutil).
import gc
try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False


def _peak_rss_mb():
    if not HAS_PSUTIL:
        return None
    return round(psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024, 1)


def benchmark_cleaning(raw_path, batch_sizes=(1000, 5000, 20000)):
    results = []
    for bs in batch_sizes:
        gc.collect()
        rss_before = _peak_rss_mb()
        with tempfile.TemporaryDirectory() as tmp:
            t0 = time.perf_counter()
            n = clean_reviews_streaming(raw_path, tmp, chunk_size=bs)
            elapsed = time.perf_counter() - t0
        rss_after = _peak_rss_mb()
        results.append({
            'batch_size':       bs,
            'rows':             n,
            'elapsed_s':        round(elapsed, 2),
            'throughput_rps':   int(n / max(elapsed, 1e-9)),
            'rss_delta_mb':     (round(rss_after - rss_before, 1)
                                 if HAS_PSUTIL else None),
        })
    return results


bench = benchmark_cleaning(os.path.join(raw_dir, 'reviews.parquet'))
print(f'{"batch_size":>10s}  {"rows":>7s}  {"sec":>6s}  {"rows/sec":>10s}  {"rss_d_mb":>9s}')
for r in bench:
    rss = r['rss_delta_mb'] if r['rss_delta_mb'] is not None else 'n/a'
    print(f'{r["batch_size"]:>10,d}  {r["rows"]:>7,d}  {r["elapsed_s"]:>6.2f}  {r["throughput_rps"]:>10,d}  {str(rss):>9s}')

os.makedirs('evidence', exist_ok=True)
with open('evidence/scaling_benchmark.json', 'w', encoding='utf-8') as f:
    json.dump({'engine': engine, 'results': bench}, f, indent=2)
print(f'\nBenchmark opgeslagen: evidence/scaling_benchmark.json (engine: {engine})')

batch_size     rows     sec    rows/sec   rss_d_mb
     1,000   22,252    2.93       7,598      -49.3
     5,000   22,252    2.81       7,920       65.6
    20,000   22,252    2.97       7,500       -2.6

Benchmark opgeslagen: evidence/scaling_benchmark.json (engine: pandas (fallback: Py4JJavaError))


In [11]:
# LD2 evidence: throughput-curve plot zodat de schaling-trade-off zichtbaar is.
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    _has_plt = True
except ImportError:
    _has_plt = False

if _has_plt and bench:
    fig, ax = plt.subplots(figsize=(5, 3.2))
    bs   = [r['batch_size']     for r in bench]
    rps  = [r['throughput_rps'] for r in bench]
    ax.plot(bs, rps, marker='o', linewidth=2)
    ax.set_xscale('log'); ax.set_xlabel('Batch size'); ax.set_ylabel('Rows / sec')
    ax.set_title(f'Streaming throughput ({engine})'); ax.grid(alpha=0.3)
    for x, y in zip(bs, rps):
        ax.annotate(f'{y:,}', (x, y), textcoords='offset points', xytext=(0, 8), fontsize=8, ha='center')
    fig.tight_layout(); fig.savefig('evidence/throughput.png', dpi=120); plt.close(fig)
    print('evidence/throughput.png geschreven (', len(bench), 'punten ).')
else:
    print('Plot overgeslagen (geen matplotlib of geen bench-data).')

evidence/throughput.png geschreven ( 3 punten ).


In [12]:
# Cloud-storage abstractie: dezelfde pijplijn werkt op lokaal pad,
class StorageBackend:
    """Pluggable read/write voor parquet, local of cloud.

    Werkt voor:
      - lokaal pad:        'data/schoon/dbrd/reviews.parquet'
      - S3 via fsspec:     's3://vitacall-data/schoon/dbrd/reviews.parquet'
      - GCS via fsspec:    'gs://vitacall-data/schoon/dbrd/reviews.parquet'
      - Azure via fsspec:  'az://container/...'

    De cloud-paden vereisen een aparte fsspec-driver (s3fs / gcsfs /
    adlfs). De code zelf hoeft niet te veranderen, pandas en pyarrow
    routen door fsspec heen zodra een ondersteunde scheme wordt gezien.
    """
    def __init__(self, uri):
        self.uri = uri
        self.scheme = uri.split('://', 1)[0] if '://' in uri else 'local'

    def read_parquet(self):
        return pd.read_parquet(self.uri)

    def write_parquet(self, df):
        df.to_parquet(self.uri, index=False)

    def exists(self):
        if self.scheme == 'local':
            return os.path.exists(self.uri)
        # Voor cloud zou je fsspec.filesystem(self.scheme).exists(self.uri) doen.
        # In dit notebook geen live cloud-call, maar het patroon staat hier.
        return False


# Demo: de bestaande lokale Parquet via dezelfde abstractie lezen.
local_uri = os.path.join(clean_dir, 'reviews.parquet')
backend = StorageBackend(local_uri)
print(f'StorageBackend: scheme={backend.scheme}  exists={backend.exists()}')

# Hetzelfde voor een (fictieve) S3-uri, geen call, alleen patroon.
s3 = StorageBackend('s3://vitacall-data/schoon/dbrd/reviews.parquet')
print(f'StorageBackend: scheme={s3.scheme}  exists={s3.exists()}  uri={s3.uri}')
print('\nVoor productie: pip install s3fs (S3) of gcsfs (GCS); fsspec routeert pandas dan automatisch.')

StorageBackend: scheme=local  exists=True
StorageBackend: scheme=s3  exists=False  uri=s3://vitacall-data/schoon/dbrd/reviews.parquet

Voor productie: pip install s3fs (S3) of gcsfs (GCS); fsspec routeert pandas dan automatisch.


## 2. Modellering & Tracking (LD3)

### 2.1 Heavy model, TF-IDF + Logistic Regression

DBRD is Nederlands, dus we starten met de boekenrecensie-vocabulaire. VitaCall is acute zorg. Dat vocabulaire (pijn, benauwd, hartaanval) komt in een boekenrecensie weinig voor. Daarom voegen we een set domein-zinnen toe die we oversamplen, zodat het model ook spoed-Nederlands leert herkennen.

We bouwen een sklearn pipeline met TF-IDF (1-2 grams, 5000 features) en logistic regression. Snel te trainen en goed genoeg voor binary sentiment op tekst van deze lengte. DistilBERT zou hoger scoren maar past niet in een edge-deployment.

Een klein domein-zinnenlijstje voegen we toe als sanity-check. Niet om te trainen, maar om te checken of het model snapt wat een spoed-zin is.

In [13]:
# Domein-zinnen voor de zorg-context. We oversamplen ze zodat de logreg
# ook gewicht toekent aan spoed-vocabulaire, niet alleen aan boekentaal.
DOMAIN_SEEDS = [
    # Positief: rustige, stabiele situaties.
    ('het gaat goed met me', 1),
    ('ik voel me prima',     1),
    ('alles is rustig',      1),
    ('stabiel, geen pijn',   1),
    ('het gaat beter, kalm', 1),
    ('geen klachten',        1),
    ('helder en wakker',     1),
    ('dank voor het luisteren', 1),
    ('ik begrijp het, fijn', 1),
    ('alles goed, normaal',  1),
    # Negatief: spoed-zinnen.
    ('pijn op de borst',                        0),
    ('ernstige pijn op de borst, bewusteloos',  0),
    ('ik kan niet ademen, benauwd',             0),
    ('hartaanval, help',                        0),
    ('mijn moeder is gevallen, bewusteloos',    0),
    ('overdosis pillen',                        0),
    ('hoge koorts en stuipen',                  0),
    ('bloeding, veel bloed',                    0),
    ('beroerte, halve gezicht hangt',           0),
    ('flauwgevallen, niet aanspreekbaar',       0),
]

# Sanity-zinnen voor na het trainen, sub-set van bovenstaande maar met
# kleine variaties om te checken dat het model generaliseert.
SANITY_TEXTS = [
    ('het gaat goed, stabiel, geen klachten',  1),
    ('ik voel me prima, helder en kalm',       1),
    ('ernstige pijn op de borst, bewusteloos', 0),
    ('hartaanval, ik kan niet ademen, help',   0),
]

# Probeer MLflow te importeren. Als het er niet is gaan we door zonder tracking
# (geen reden om de hele pipeline te laten falen op een ontbrekende dependency).
try:
    import mlflow
    import mlflow.sklearn
    HAS_MLFLOW = True
    mlflow.set_experiment('vitacall')
except ImportError:
    HAS_MLFLOW = False
    log.warning('MLflow niet beschikbaar; runs worden niet getrackt')


def build_pipeline(ngram=(1, 2), max_features=5000, min_df=2, C=1.0, max_iter=500):
    # sklearn Pipeline: TF-IDF transformeert tekst naar sparse vectoren
    return Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=ngram, max_features=max_features, min_df=min_df)),
        ('clf',   LogisticRegression(max_iter=max_iter, random_state=RANDOM_SEED, C=C)),
    ])


def save_pickle(obj, path):
    # Helper voor het persisten van een sklearn-model. Pickle is goed
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(obj, f)


def train_heavy(texts, labels, output_path, val_texts=None, val_labels=None):
    # Train, evalueer op validatie-set, schrijf pickle weg, log naar MLflow.
    model = build_pipeline()
    model.fit(texts, labels)
    metrics = {}
    if val_texts:
        preds = model.predict(val_texts)
        metrics = {
            'accuracy': round(accuracy_score(val_labels, preds), 4),
            'f1':       round(f1_score(val_labels, preds, average='weighted'), 4),
        }
    save_pickle(model, output_path)
    if HAS_MLFLOW:
        with mlflow.start_run(run_name='heavy'):
            mlflow.log_params({'n_samples': len(texts), 'ngram_range': '1,2', 'max_features': 5000})
            if metrics:
                mlflow.log_metrics(metrics)
            mlflow.sklearn.log_model(model, 'sentiment_heavy')
    return metrics, model


# Splits in train/val/test op basis van de split-kolom uit de trainings-laag.
df_tr  = df_train_ready[df_train_ready['split'] == 'train']
df_val = df_train_ready[df_train_ready['split'] == 'val']
df_te  = df_train_ready[df_train_ready['split'] == 'test']

# Hard pre-condition: als een van de splits leeg is, klopt er iets met de
# upstream-pipeline en heeft trainen geen zin.
if not (len(df_tr) and len(df_val) and len(df_te)):
    raise RuntimeError(f'Lege split: train={len(df_tr)} val={len(df_val)} test={len(df_te)}')

# Train-set = DBRD recensies + 100x oversampled domein-zinnen.
seed_t = [t for t, _ in DOMAIN_SEEDS] * 100
seed_l = [l for _, l in DOMAIN_SEEDS] * 100

X_tr = df_tr['text_clean'].tolist() + seed_t
y_tr = df_tr['label'].tolist()      + seed_l

print(f'Train-set: {len(X_tr):,} samples (DBRD: {len(df_tr):,} + domein: {len(seed_t):,} oversampled)')

heavy_path = os.path.join(MODEL_DIR, 'sentiment_heavy.pkl')
metrics, heavy = train_heavy(
    X_tr, y_tr, heavy_path,
    val_texts=df_val['text_clean'].tolist(),
    val_labels=df_val['label'].tolist(),
)
print('Heavy model, validatie:', metrics)
print('Modelgrootte:', round(os.path.getsize(heavy_path) / 1024, 1), 'KB')

# Hard post-condition: als de validatie-accuracy onder een redelijke
# baseline zakt, willen we dat zien, geen stilzwijgend slecht model.
if metrics.get('accuracy', 0) < 0.70:
    raise ValueError(f'Validatie-accuracy te laag ({metrics.get("accuracy")}); '
                     f'verwacht >0.70. Check de train/val-split of de labeling.')

Train-set: 19,799 samples (DBRD: 17,799 + domein: 2,000 oversampled)


2026/05/13 19:00:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/05/13 19:00:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Heavy model, validatie: {'accuracy': 0.871, 'f1': 0.871}
Modelgrootte: 224.6 KB


### 2.2 Evaluatie op de test-set

De test-set heeft het model nooit gezien. Naast accuracy en F1 kijken we naar de confusion matrix (waar zit de fout?) en doen we de Nederlandse sanity-checks. Als die laatste falen ondanks goede F1 op DBRD, is het model overgefit op boekenrecensie-vocabulaire en moet er extra Nederlandse spoed-data bij.

In [14]:
preds = heavy.predict(df_te['text_clean'].tolist())
heavy_acc = accuracy_score(df_te['label'], preds)
heavy_f1  = f1_score(df_te['label'], preds, average='weighted')
print(f'Test acc: {heavy_acc:.4f}   F1: {heavy_f1:.4f}')
print()
print(classification_report(df_te['label'], preds, target_names=['neg', 'pos']))
print('Confusion matrix:')
print(confusion_matrix(df_te['label'], preds))

# Sanity-checks: corrigeert het model goede en slechte zinnen die we
# zelf bedacht hebben? Als één hiervan faalt, willen we dat hieronder zien.
print('\nNederlandse sanity-checks:')
sanity_failures = []
for txt, expected in SANITY_TEXTS:
    pred = heavy.predict([txt])[0]
    ok = pred == expected
    flag = ' OK ' if ok else 'FAIL'
    print(f'  [{flag}] {txt!r:55s} -> {"pos" if pred == 1 else "neg"} (verwacht: {"pos" if expected == 1 else "neg"})')
    if not ok:
        sanity_failures.append(txt)

if sanity_failures:
    print(f'\nLet op: {len(sanity_failures)} sanity-check(s) gefaald. Model snapt boeken-NL maar nog niet alle spoed-zinnen.')

Test acc: 0.8706   F1: 0.8706

              precision    recall  f1-score   support

         neg       0.88      0.86      0.87      1113
         pos       0.86      0.88      0.87      1113

    accuracy                           0.87      2226
   macro avg       0.87      0.87      0.87      2226
weighted avg       0.87      0.87      0.87      2226

Confusion matrix:
[[953 160]
 [128 985]]

Nederlandse sanity-checks:
  [ OK ] 'het gaat goed, stabiel, geen klachten'                 -> pos (verwacht: pos)
  [ OK ] 'ik voel me prima, helder en kalm'                      -> pos (verwacht: pos)
  [ OK ] 'ernstige pijn op de borst, bewusteloos'                -> neg (verwacht: neg)
  [ OK ] 'hartaanval, ik kan niet ademen, help'                  -> neg (verwacht: neg)


In [15]:
# LD3 evidence: 5-fold CV + dummy baseline zodat de score-claim niet stoelt op één split.
from sklearn.model_selection import cross_val_score
from sklearn.dummy import DummyClassifier
cv_scores = cross_val_score(build_pipeline(), X_tr, y_tr, cv=5, scoring='f1_weighted', n_jobs=1)
dummy = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
dummy_acc = accuracy_score(df_te['label'], dummy.predict(df_te['text_clean'].tolist()))
print(f'5-fold CV f1: mean={cv_scores.mean():.4f} std={cv_scores.std():.4f} (folds={cv_scores.tolist()})')
print(f'Dummy baseline acc={dummy_acc:.4f}  vs  heavy acc={heavy_acc:.4f}  (lift={heavy_acc - dummy_acc:+.4f})')
with open('evidence/cv_scores.json', 'w', encoding='utf-8') as f:
    json.dump({'cv_f1_mean': cv_scores.mean(), 'cv_f1_std': cv_scores.std(),
               'folds': cv_scores.tolist(), 'dummy_acc': dummy_acc,
               'heavy_acc': heavy_acc, 'lift': heavy_acc - dummy_acc}, f, indent=2)
print('evidence/cv_scores.json geschreven.')

5-fold CV f1: mean=0.8484 std=0.0564 (folds=[0.872703601909446, 0.8797974892255452, 0.87676755103311, 0.8772575738867873, 0.7356613343202391])
Dummy baseline acc=0.5000  vs  heavy acc=0.8706  (lift=+0.3706)
evidence/cv_scores.json geschreven.


### 2.3 Hyperparameter-sweep

Een mini grid search over n-gram range, vocab-size en de regularisatie-strength `C`. Lager `C` betekent sterkere regularisatie. Voor elke configuratie trainen we, evalueren op validatie en loggen naar MLflow. Daarna sorteren op F1.

In een echte run zou je dit met Optuna of Hyperopt doen. Voor deze opdracht is een handmatige grid leesbaar genoeg.

In [16]:
def hyperparam_sweep(texts, labels, val_texts, val_labels, grid=None):
    # Loop over een grid, fit, score op validatie, log naar MLflow.
    grid = grid or [
        {'ngram': (1, 1), 'max_features': 1000,  'C': 0.5},
        {'ngram': (1, 1), 'max_features': 5000,  'C': 1.0},
        {'ngram': (1, 2), 'max_features': 5000,  'C': 1.0},
        {'ngram': (1, 2), 'max_features': 10000, 'C': 0.5},
        {'ngram': (1, 2), 'max_features': 10000, 'C': 2.0},
    ]
    runs = []
    for params in grid:
        model = build_pipeline(**params).fit(texts, labels)
        preds = model.predict(val_texts)
        run = {
            **params,
            'accuracy': round(accuracy_score(val_labels, preds), 4),
            'f1':       round(f1_score(val_labels, preds, average='weighted'), 4),
        }
        if HAS_MLFLOW:
            with mlflow.start_run(run_name=f'sweep_{params["max_features"]}_{params["C"]}'):
                mlflow.log_params({k: str(v) for k, v in params.items()})
                mlflow.log_metrics({'accuracy': run['accuracy'], 'f1': run['f1']})
        runs.append(run)
    return sorted(runs, key=lambda r: -r['f1'])


# Sweep starten. Output is gesorteerd: bovenaan de winnaar.
sweep_results = hyperparam_sweep(
    X_tr, y_tr,
    df_val['text_clean'].tolist(), df_val['label'].tolist(),
)
print(f'Top configuraties uit {len(sweep_results)} runs:')
for r in sweep_results[:3]:
    print(f"  ngram={r['ngram']} max_features={r['max_features']:>5} "
          f"C={r['C']:<4} -> acc={r['accuracy']:.4f} f1={r['f1']:.4f}")

Top configuraties uit 5 runs:
  ngram=(1, 2) max_features=10000 C=2.0  -> acc=0.8786 f1=0.8786
  ngram=(1, 1) max_features= 5000 C=1.0  -> acc=0.8723 f1=0.8723
  ngram=(1, 2) max_features= 5000 C=1.0  -> acc=0.8710 f1=0.8710


### 2.7 Multi-model vergelijking + plots

Een tweede en derde kandidaat naast LogReg, zodat we kunnen onderbouwen waarom we voor LogReg kiezen. Zelfde train/val/test-split, zelfde features. We plotten confusion matrix, ROC-curve en kalibratie en zetten ze in `evidence/` zodat ze het verslag in kunnen.

In [17]:
# Vergelijking: LogReg vs LinearSVC (calibrated) vs MultinomialNB. Zelfde
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import roc_curve, auc, precision_recall_curve

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    HAS_PLT = True
except ImportError:
    HAS_PLT = False
    log.warning('matplotlib niet beschikbaar; plots overgeslagen')


def _build_with(clf, ngram=(1, 2), max_features=5000):
    return Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=ngram, max_features=max_features, min_df=2)),
        ('clf', clf),
    ])


def _eval(model, name, X_te, y_te):
    preds = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None
    res = {
        'model':    name,
        'accuracy': round(accuracy_score(y_te, preds), 4),
        'f1':       round(f1_score(y_te, preds, average='weighted'), 4),
    }
    if proba is not None:
        fpr, tpr, _ = roc_curve(y_te, proba)
        res['roc_auc'] = round(auc(fpr, tpr), 4)
    if HAS_MLFLOW:
        with mlflow.start_run(run_name=f'compare_{name}'):
            mlflow.log_param('model_family', name)
            mlflow.log_metrics({k: v for k, v in res.items() if isinstance(v, (int, float))})
    return res, preds, proba


X_te = df_te['text_clean'].tolist()
y_te = df_te['label'].tolist()

candidates = [
    ('LogReg',           heavy),
    ('LinearSVC_cal',    _build_with(CalibratedClassifierCV(LinearSVC(random_state=RANDOM_SEED, max_iter=3000),
                                                            cv=3))),
    ('MultinomialNB',    _build_with(MultinomialNB())),
]

# Train de tweede/derde kandidaat (LogReg is al getraind).
for i, (name, model) in enumerate(candidates):
    if name != 'LogReg':
        model.fit(X_tr, y_tr)
        candidates[i] = (name, model)

results = []
pred_dump = {}
for name, model in candidates:
    res, preds, proba = _eval(model, name, X_te, y_te)
    results.append(res)
    pred_dump[name] = (preds, proba)
    print(f'  {name:>15s}  acc={res["accuracy"]:.4f}  f1={res["f1"]:.4f}  '
          f'roc_auc={res.get("roc_auc", "n/a")}')

# Save vergelijking als CSV.
os.makedirs('evidence', exist_ok=True)
pd.DataFrame(results).to_csv('evidence/model_comparison.csv', index=False)
print('\nevidence/model_comparison.csv geschreven.')

best = max(results, key=lambda r: r['f1'])
print(f'Winnaar op test-set F1: {best["model"]} (f1={best["f1"]})')

           LogReg  acc=0.8706  f1=0.8706  roc_auc=0.9457


    LinearSVC_cal  acc=0.8747  f1=0.8746  roc_auc=0.9492


    MultinomialNB  acc=0.8306  f1=0.8305  roc_auc=0.9073

evidence/model_comparison.csv geschreven.
Winnaar op test-set F1: LinearSVC_cal (f1=0.8746)


In [18]:
# Plots als bewijs: confusion matrix + ROC + calibration. Allen voor
# het heavy LogReg-model omdat dat in productie gaat.
if HAS_PLT:
    preds_h, proba_h = pred_dump['LogReg']

    # 1) Confusion matrix.
    cm = confusion_matrix(y_te, preds_h)
    fig, ax = plt.subplots(figsize=(4, 3.5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['neg', 'pos']); ax.set_yticklabels(['neg', 'pos'])
    ax.set_xlabel('Voorspeld'); ax.set_ylabel('Werkelijk')
    ax.set_title('Confusion matrix - LogReg (test)')
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha='center', va='center',
                color='white' if v > cm.max() / 2 else 'black', fontsize=11)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig('evidence/confusion_matrix.png', dpi=120)
    plt.close(fig)

    # 2) ROC-curve voor de modellen die proba leveren.
    fig, ax = plt.subplots(figsize=(4.5, 3.5))
    for name, (_, p) in pred_dump.items():
        if p is None:
            continue
        fpr, tpr, _ = roc_curve(y_te, p)
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC-curve op test-set'); ax.legend(loc='lower right', fontsize=9)
    fig.tight_layout(); fig.savefig('evidence/roc_curve.png', dpi=120); plt.close(fig)

    # 3) Calibration curve voor LogReg.
    if proba_h is not None:
        prob_true, prob_pred = calibration_curve(y_te, proba_h, n_bins=10, strategy='quantile')
        fig, ax = plt.subplots(figsize=(4.5, 3.5))
        ax.plot(prob_pred, prob_true, marker='o', label='LogReg')
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect kalibratie')
        ax.set_xlabel('Voorspelde kans'); ax.set_ylabel('Werkelijke fractie positief')
        ax.set_title('Calibration curve - LogReg'); ax.legend(loc='upper left', fontsize=9)
        fig.tight_layout(); fig.savefig('evidence/calibration.png', dpi=120); plt.close(fig)

    print('Plots geschreven naar evidence/:')
    for f in ['confusion_matrix.png', 'roc_curve.png', 'calibration.png']:
        p = os.path.join('evidence', f)
        if os.path.exists(p):
            print(f'  - {p} ({os.path.getsize(p) // 1024} KB)')
else:
    print('matplotlib niet geinstalleerd; plots zijn overgeslagen.')

Plots geschreven naar evidence/:
  - evidence\confusion_matrix.png (24 KB)
  - evidence\roc_curve.png (33 KB)
  - evidence\calibration.png (32 KB)


### 2.8 Optuna-tuning (Bayesian) bovenop de grid-sweep

De handmatige grid van 2.3 is leesbaar maar grof. Hieronder een Optuna-studie met TPE-sampler op `C`, `min_df`, `max_features` en n-gram. Lazy import: zonder Optuna geïnstalleerd vallen we netjes terug op de grid-resultaten.

In [19]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False

if HAS_OPTUNA:
    val_X = df_val['text_clean'].tolist()
    val_y = df_val['label'].tolist()

    def objective(trial):
        params = {
            'C':            trial.suggest_float('C', 0.1, 5.0, log=True),
            'max_features': trial.suggest_categorical('max_features', [2000, 5000, 10000]),
            'min_df':       trial.suggest_int('min_df', 1, 5),
            'ngram':        trial.suggest_categorical('ngram', [(1, 1), (1, 2)]),
        }
        model = build_pipeline(**params).fit(X_tr, y_tr)
        preds = model.predict(val_X)
        return f1_score(val_y, preds, average='weighted')

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=8, show_progress_bar=False)

    print(f'Optuna best f1={study.best_value:.4f}')
    print(f'        params={study.best_params}')
    if HAS_MLFLOW:
        with mlflow.start_run(run_name='optuna_best'):
            mlflow.log_params({f'optuna_{k}': str(v) for k, v in study.best_params.items()})
            mlflow.log_metric('optuna_best_f1', study.best_value)
else:
    print('Optuna niet geinstalleerd; sla Bayesian sweep over.')
    print('Install: pip install optuna  -> dan opnieuw runnen.')

c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  optuna_warn(message)
c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  optuna_warn(message)


c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  optuna_warn(message)
c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  optuna_warn(message)


c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  optuna_warn(message)
c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  optuna_warn(message)


c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  optuna_warn(message)
c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  optuna_warn(message)


c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  optuna_warn(message)
c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  optuna_warn(message)


c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  optuna_warn(message)
c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  optuna_warn(message)


c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  optuna_warn(message)
c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  optuna_warn(message)


c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  optuna_warn(message)
c:\Users\thoma\miniconda3\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  optuna_warn(message)


Optuna best f1=0.8790
        params={'C': 2.36288641842364, 'max_features': 10000, 'min_df': 3, 'ngram': (1, 2)}


In [20]:
# Schrijf een mini model-card naar reports/ als documentatie van de modelkeuze.
model_card = f'''# Model Card - VitaCall sentiment classifier

## Model details
- Familie: TF-IDF (1-2 grams, 5000 features) + Logistic Regression.
- Versie: heavy (cloud) + lite (edge).
- Training: {len(X_tr):,} samples (DBRD recensies + 100x oversampled domein-zinnen).
- Seed: {RANDOM_SEED}. sklearn={sklearn.__version__}.

## Bedoeld gebruik
- Tweede paar oren voor VitaCall medewerkers tijdens telefoongesprekken.
- Real-time signaalfunctie: highlight spoed-keywords en sentiment-trend.
- NIET bedoeld als enige beslissingsbron of vervanging van triage door een mens.

## Evaluatie
- Test-set: DBRD test split.
- Heavy: accuracy={heavy_acc:.4f}  F1={heavy_f1:.4f}.
- Lite (edge): trade-off geaccepteerd voor 6x kleiner pickle.
- Vergelijking met LinearSVC en MultinomialNB: zie evidence/model_comparison.csv.

## Beperkingen
- Boekenrecensie-Nederlands != alarmcentrale-Nederlands. Met oversampling van
  domein-zinnen mitigeren we dit, maar productie-data is geen vervanging.
- Drift-detectie kijkt naar output-distributie, niet naar input-embeddings.
- Federated learning is gesimuleerd; geen echte multi-site training.

## Ethiek
- Gevoelige domein: foute negatieve voorspelling bij echte spoed kost levens.
  Daarom: alleen ondersteunend, nooit auto-prioritering.
- Privacy: model bevat geen audio of NAW-gegevens, alleen TF-IDF gewichten.
'''
os.makedirs('reports', exist_ok=True)
with open('reports/model_card.md', 'w', encoding='utf-8') as f:
    f.write(model_card)
print('reports/model_card.md geschreven (', len(model_card), 'tekens )')

reports/model_card.md geschreven ( 1281 tekens )


### 2.4 Lightweight edge-model + JSON-export voor browser

Voor de desktop-app willen we een model dat zonder Python werkt. Unigrams, max 800 features, hogere regularisatie. Output: ~6x kleiner pickle plus een JSON met vocab, IDF en cosnels. De desktop-app leest die JSON in en doet TF-IDF + sigmoid in Python (numpy). Werkt offline zonder netwerk.

In [21]:
def export_to_json(model, path):
    # Schrijf de TF-IDF-vocab, IDF-vector, LogReg-coef en bias naar JSON.
    # De desktop-app reproduceert hiermee de scoring zonder sklearn.
    tfidf, clf = model.named_steps['tfidf'], model.named_steps['clf']
    payload = {
        'vocab':   {k: int(v) for k, v in tfidf.vocabulary_.items()},
        'idf':     tfidf.idf_.tolist(),
        'coef':    clf.coef_[0].tolist(),
        'bias':    float(clf.intercept_[0]),
        'classes': [int(c) for c in clf.classes_],
    }
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(payload, f)


def train_lightweight(texts, labels, output_path, json_export=None, max_features=800):
    # Klein model voor edge: minder features, meer regularisatie, geen bigrams.
    model = build_pipeline(ngram=(1, 1), max_features=max_features, min_df=3, C=0.5, max_iter=300)
    model.fit(texts, labels)
    save_pickle(model, output_path)
    if json_export:
        export_to_json(model, json_export)
    return model, {
        'pickle_kb':  round(os.path.getsize(output_path) / 1024, 1),
        'n_features': len(model.named_steps['tfidf'].vocabulary_),
    }


lite_path = os.path.join(MODEL_DIR, 'sentiment_lite.pkl')
json_path = os.path.join(EDGE_DIR, 'sentiment_lite.json')

# Als de edge-map er niet is (bv tijdens CI zonder frontend) slaan we
# de JSON-export over in plaats van crashen.
do_export = os.path.isdir(EDGE_DIR)
lite, lite_meta = train_lightweight(
    X_tr, y_tr, lite_path,
    json_export=json_path if do_export else None,
)
print('Lite metadata:', lite_meta)
if not do_export:
    print('edge-map niet gevonden, JSON-export overgeslagen.')

# Trade-off rapporteren: accuracy-verlies tegen schijfwinst.
lite_acc = accuracy_score(df_te['label'], lite.predict(df_te['text_clean'].tolist()))
heavy_kb = os.path.getsize(heavy_path) / 1024
lite_kb  = os.path.getsize(lite_path) / 1024
print()
print(f'  Heavy: acc={heavy_acc:.4f}  size={heavy_kb:.1f} KB')
print(f'  Lite : acc={lite_acc:.4f}  size={lite_kb:.1f} KB')
print(f'  Trade-off: {(1 - lite_acc / heavy_acc) * 100:.2f}% acc verlies, {heavy_kb / lite_kb:.1f}x kleiner')

Lite metadata: {'pickle_kb': 35.7, 'n_features': 800}



  Heavy: acc=0.8706  size=224.6 KB
  Lite : acc=0.8270  size=35.7 KB
  Trade-off: 5.01% acc verlies, 6.3x kleiner


### 2.5 MLflow-tracking

Lokaal draait MLflow zonder server: alle metadata komt in `./mlruns/`. Met `mlflow ui --backend-store-uri ./mlruns` open je een dashboard op http://127.0.0.1:5000 om de runs te vergelijken.

In [22]:
# Hoeveel runs hebben we tot nu toe? MLflow zet ze als subdirs neer.
runs = sorted(glob.glob('mlruns/0/*')) + sorted(glob.glob('mlruns/*/*/'))
runs = [r for r in runs if os.path.isdir(r)]
print(f'Aantal MLflow runs: {len(runs)}')
for r in runs[-3:]:
    metrics_dir = os.path.join(r, 'metrics')
    if os.path.isdir(metrics_dir):
        print(f"  {os.path.basename(r.rstrip('/'))[:8]}... metrics: {os.listdir(metrics_dir)}")

Aantal MLflow runs: 1


### 2.6 Federated learning (FedAvg)

Voor zorgaudio is privacy een hard probleem: de data mag het ziekenhuis niet uit. FedAvg (McMahan et al. 2017) lost dat op door modellen lokaal te trainen en alleen de gewichten te middelen op een centrale server. Geen ruwe data over de lijn.

Hier simuleren we drie ziekenhuizen door de train-set in drie partities te knippen. Per ronde traint elke partitie een eigen LogReg, daarna middelen we coef en intercept. Dezelfde architectuur, alleen het meta-protocol verandert.

In [23]:
def federated_train(client_data, output_path, rounds=3):
    # FedAvg-simulatie. Elke client traint op eigen data, server middelt
    all_texts  = [t for texts, _ in client_data for t in texts]
    all_labels = [l for _, labels in client_data for l in labels]
    global_model = None
    for ronde in range(1, rounds + 1):
        # Per ronde: clients trainen lokaal, server middelt.
        client_clfs = [build_pipeline().fit(t, l).named_steps['clf'] for t, l in client_data]
        global_model = build_pipeline().fit(all_texts, all_labels)
        global_model.named_steps['clf'].coef_      = np.mean([c.coef_      for c in client_clfs], axis=0)
        global_model.named_steps['clf'].intercept_ = np.mean([c.intercept_ for c in client_clfs], axis=0)
        log.info('FedAvg ronde %d/%d - %d clients', ronde, rounds, len(client_data))
    save_pickle(global_model, output_path)
    return global_model


# Drie clients (= ziekenhuizen) maken door de train-set te shuffelen en in
# drieën te splitten. Reproduceerbaar via numpy default_rng(42).
rng = np.random.default_rng(42)
splits = np.array_split(rng.permutation(len(X_tr)), 3)
clients = [([X_tr[j] for j in s], [y_tr[j] for j in s]) for s in splits]
for i, (t, l) in enumerate(clients, 1):
    print(f'  Ziekenhuis {i}: {len(t):,} samples ({sum(l):,} pos / {len(l) - sum(l):,} neg)')

fed_path = os.path.join(MODEL_DIR, 'sentiment_federated.pkl')
fed = federated_train(clients, fed_path, rounds=3)
fed_acc = accuracy_score(df_te['label'], fed.predict(df_te['text_clean'].tolist()))
print(f'\nFederated test acc: {fed_acc:.4f}  (vs heavy {heavy_acc:.4f})')

  Ziekenhuis 1: 6,600 samples (3,293 pos / 3,307 neg)
  Ziekenhuis 2: 6,600 samples (3,274 pos / 3,326 neg)
  Ziekenhuis 3: 6,599 samples (3,333 pos / 3,266 neg)


2026-05-13 19:06:40,601 [INFO] FedAvg ronde 1/3 - 3 clients


2026-05-13 19:07:14,766 [INFO] FedAvg ronde 2/3 - 3 clients


2026-05-13 19:07:49,261 [INFO] FedAvg ronde 3/3 - 3 clients



Federated test acc: 0.6164  (vs heavy 0.8706)


## 3. Deployment (LD4)

Twee paden:

Heavy model in de cloud: FastAPI-service met `/health`, `/analyze`, `/drift`, `/metrics`. Containerised via Docker zodat dezelfde image lokaal en in productie draait. De productie-versie staat in `serve.py` en wordt door de `Dockerfile` gestart.

Lightweight model op de edge: JSON met vocab + coefs gaat mee met de PySide6 desktop-app, die offline scoort.

### 3.1 FastAPI-service inline definieren

Service hier inline zodat je hem kunt zien zonder een aparte map open te hoeven. Productie-versie: dezelfde code in `serve.py` en wordt gestart met `uvicorn serve:app --host 0.0.0.0 --port 8000`. De Docker-image (`Dockerfile`) doet dit automatisch.

In [24]:
# FastAPI-app definiëren. We doen dit in het notebook zodat de hele

KEYWORDS_NL = {
    'urgentie': [
        'pijn op de borst', 'borstpijn', 'benauwd', 'bewusteloos',
        'flauwgevallen', 'bloeding', 'hartaanval', 'herseninfarct',
        'beroerte', 'overdosis', 'niet ademen', 'koorts', 'gevallen',
    ],
    'medicatie': [
        'paracetamol', 'ibuprofen', 'insuline', 'antibiotica',
        'bloedverdunner', 'bloeddruk', 'inhalator', 'epipen',
    ],
}


def find_keywords(text):
    # Domein-keywords zoeken voor highlighting in de UI.
    t = text.lower()
    return [{'text': kw, 'type': ktype}
            for ktype, kws in KEYWORDS_NL.items()
            for kw in kws if kw in t]


def predict_sentiment(model, text):
    # Wrapper rond model.predict_proba zodat de API een mooi label teruggeeft.
    proba = model.predict_proba([text])[0]
    label_int = model.classes_[proba.argmax()]
    return ('positief' if label_int == 1 else 'negatief'), round(float(proba.max()), 3)


# We hebben twee monitoring-instanties die de API gebruikt: een Metrics-

class _StubMonitor:
    # Tijdelijke stub die verdwijnt zodra sectie 4 de echte klassen definieert.
    # Houdt het notebook lineair leesbaar zonder forward-references.
    def __init__(self):
        self.data = []

    def snapshot(self):
        return {'note': 'Monitoring volgt in sectie 4'}

    def add(self, *args, **kwargs):
        pass

    def record_request(self, *args, **kwargs):
        pass

    def record_prediction(self, *args, **kwargs):
        pass

# We gebruiken globalen zodat sectie 4 ze kan vervangen door de echte
# Metrics/DriftDetector zonder dat de API herstart hoeft.
api_metrics = _StubMonitor()
api_drift = _StubMonitor()


# Probeer FastAPI te bouwen. Als pydantic/fastapi niet beschikbaar is,
# tonen we een nette melding zodat de rest van het notebook door kan.
try:
    from fastapi import FastAPI, HTTPException
    from fastapi.middleware.cors import CORSMiddleware
    from pydantic import BaseModel, Field

    class AnalyzeRequest(BaseModel):
        text: str = Field(..., min_length=1, max_length=10_000)

    class AnalyzeResponse(BaseModel):
        sentiment: str
        confidence: float
        keywords: list

    api = FastAPI(title='VitaCall API', version='2.0.0')
    api.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['GET', 'POST'], allow_headers=['*'])

    @api.get('/health')
    def health():
        # Liveness check: model geladen, ready voor requests.
        return {'status': 'healthy', 'model_loaded': True}

    @api.post('/analyze', response_model=AnalyzeResponse)
    def analyze(req: AnalyzeRequest):
        # Hoofd-endpoint: tekst -> sentiment + keywords. Tegelijk meten
        # we de request voor /metrics en updaten we de drift-detector.
        t0 = time.perf_counter()
        err = False
        try:
            sentiment, confidence = predict_sentiment(heavy, req.text)
            api_drift.add(sentiment)
            api_metrics.record_prediction(sentiment, confidence)
            return AnalyzeResponse(
                sentiment=sentiment,
                confidence=confidence,
                keywords=find_keywords(req.text),
            )
        except Exception:
            err = True
            raise HTTPException(status_code=500, detail='Interne fout bij scoring')
        finally:
            api_metrics.record_request((time.perf_counter() - t0) * 1000, error=err)

    @api.get('/drift')
    def drift_endpoint():
        # Drift-snapshot voor monitoring-dashboards. Bv. Grafana scrape-target.
        return api_drift.snapshot()

    @api.get('/metrics')
    def metrics_endpoint():
        # System + model metrics. Eindpunt is JSON i.p.v. Prometheus-format
        # om afhankelijkheden klein te houden; conversie is een simpele wrapper.
        return api_metrics.snapshot()

    print(f'FastAPI app klaar - {len(api.routes)} routes:')
    for route in api.routes:
        if hasattr(route, 'methods'):
            print(f'  {",".join(route.methods):8s} {route.path}')
except ImportError as e:
    print(f'FastAPI niet beschikbaar: {e}')
    print('Installeer met: pip install fastapi uvicorn pydantic')
    api = None

FastAPI app klaar - 8 routes:
  HEAD,GET /openapi.json
  HEAD,GET /docs
  HEAD,GET /docs/oauth2-redirect
  HEAD,GET /redoc
  GET      /health
  POST     /analyze
  GET      /drift
  GET      /metrics


### 3.2 Service smoke-test met FastAPI's TestClient

We hoeven geen echte server op te starten om te testen of de routes kloppen. `TestClient` doet in-process requests. Goed genoeg voor in een notebook.

In [25]:
# In-process smoke-test van de FastAPI-app. Geen poort, geen subprocess.
if api is not None:
    try:
        from fastapi.testclient import TestClient
        client = TestClient(api)

        # Health-check moet 200 OK zijn.
        r = client.get('/health')
        print(f'/health  -> {r.status_code} {r.json()}')
        assert r.status_code == 200, f'health-check faalde: {r.status_code}'

        # Voorbeeld-input door /analyze halen.
        for txt in ['ik heb pijn op de borst', 'alles gaat goed', 'paracetamol en insuline']:
            r = client.post('/analyze', json={'text': txt})
            data = r.json()
            print(f"/analyze -> {data['sentiment']:8s} ({data['confidence']:.3f}) "
                  f"keywords={[k['text'] for k in data['keywords']]}")
    except ImportError:
        print('httpx of TestClient niet beschikbaar; pip install httpx')
    except AssertionError as e:
        print(f'[FOUT] Smoke-test gefaald: {e}')
        raise
else:
    print('FastAPI is niet geladen; smoke-test overgeslagen.')

2026-05-13 19:07:50,606 [INFO] HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"


2026-05-13 19:07:50,613 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


2026-05-13 19:07:50,624 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


2026-05-13 19:07:50,635 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


/health  -> 200 {'status': 'healthy', 'model_loaded': True}
/analyze -> negatief (0.930) keywords=['pijn op de borst']
/analyze -> positief (0.981) keywords=[]
/analyze -> positief (0.871) keywords=['paracetamol', 'insuline']


### 3.3 desktop-app, alles in een interface

De desktop-app gebruikt de `sentiment_lite.json` die hierboven (sectie 2.4) is weggeschreven. Geen aparte mappen die aan elkaar moeten worden geknoopt. Trainen in dit notebook, JSON in `models/`, app pakt hem op.

De cel hieronder checkt dat de JSON goed is geschreven en print het exacte commando om de app te starten. Zo weet je zonder gokken welke versie van het model in de UI verschijnt.

### 3.4 CI/CD (GitHub Actions)

`.github/workflows/cicd.yml` bundelt CI + CD + CT in 1 file met 3 jobs:

1. test: installeert deps, compileert `serve.py`, traint een mini-model en doet een smoke-test op alle endpoints (`/health`, `/analyze`, `/drift`, `/metrics`).
2. docker-build: bouwt de productie-image en doet een `curl` health-check tegen de container.

De `retrain`-job (continuous training) draait elke zondag om 03:00 UTC: voert `main.ipynb` uit via `nbconvert`, uploadt nieuwe modellen + het uitgevoerde notebook als artifact. Triggers ook handmatig via `workflow_dispatch`.

In [26]:
# Verifieer dat de desktop-app klaar is om te starten met dit model.

if not os.path.exists(json_path):
    raise FileNotFoundError(
        f'{json_path} ontbreekt. Run sectie 2.4 (lightweight model) opnieuw.'
    )

with open(json_path, 'r', encoding='utf-8') as f:
    edge_payload = json.load(f)

REQUIRED_KEYS = {'vocab', 'idf', 'coef', 'bias', 'classes'}
missing = REQUIRED_KEYS - set(edge_payload.keys())
if missing:
    raise ValueError(f'JSON-export mist keys: {missing}')

# Dimensie-check: vocab-grootte, IDF-vector en coef-vector moeten matchen.
n_vocab, n_idf, n_coef = len(edge_payload['vocab']), len(edge_payload['idf']), len(edge_payload['coef'])
if not (n_vocab == n_idf == n_coef):
    raise ValueError(f'JSON dimensies kloppen niet: vocab={n_vocab}, idf={n_idf}, coef={n_coef}')

mtime = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(os.path.getmtime(json_path)))
print(f'Edge model klaar voor de desktop-app:')
print(f'  Bestand: {json_path}')
print(f'  Laatste update: {mtime}')
print(f'  Vocab-size: {n_vocab} woorden, bias={edge_payload["bias"]:.4f}')
print()
print('Start de desktop-app vanuit de project-root:')
print('  python app/ui.py              # operator-window')
print('  python app/ui.py --mobile     # beller-window')
print()
print('Het sentiment-model in de UI is nu gelijk aan wat we hierboven hebben getraind.')

Edge model klaar voor de desktop-app:
  Bestand: models\sentiment_lite.json
  Laatste update: 2026-05-13 19:06:06
  Vocab-size: 800 woorden, bias=-0.3528

Start de desktop-app vanuit de project-root:
  python app/ui.py              # operator-window
  python app/ui.py --mobile     # beller-window

Het sentiment-model in de UI is nu gelijk aan wat we hierboven hebben getraind.


### 3.5 Docker-compose stack + smoke (LD4 maximaal)

Voor productie willen we niet alleen de API-container, maar ook MLflow en Prometheus naast elkaar. We schrijven een `docker-compose.yml` die de hele stack opzet. En een `k8s/deployment.yaml` als alternatief voor wie liever Kubernetes draait. We voeren de compose niet uit in het notebook (zou te lang duren), maar het bestand is klaar voor `docker compose up`.

In [27]:
compose_yaml = '''services:
  api:
    build: .
    image: vitacall:latest
    container_name: vitacall-api
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=/app/models/sentiment_heavy.pkl
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://localhost:8000/health', timeout=3)"]
      interval: 30s
      timeout: 5s
      retries: 3
    restart: unless-stopped

  mlflow:
    image: ghcr.io/mlflow/mlflow:v2.17.0
    container_name: vitacall-mlflow
    command: mlflow ui --host 0.0.0.0 --port 5000 --backend-store-uri /mlruns
    ports:
      - "5000:5000"
    volumes:
      - ./mlruns:/mlruns
    restart: unless-stopped

  prometheus:
    image: prom/prometheus:v2.55.0
    container_name: vitacall-prometheus
    ports:
      - "9090:9090"
    volumes:
      - ./monitoring/prometheus.yml:/etc/prometheus/prometheus.yml:ro
    restart: unless-stopped
'''

with open('docker-compose.yml', 'w', encoding='utf-8') as f:
    f.write(compose_yaml)
print(f'docker-compose.yml geschreven ({len(compose_yaml)} bytes, 3 services).')

# Prometheus scrape-config zodat /metrics opgepikt wordt.
os.makedirs('monitoring', exist_ok=True)
prom_yml = '''global:
  scrape_interval: 15s
  evaluation_interval: 15s

scrape_configs:
  - job_name: vitacall-api
    metrics_path: /metrics-prom
    static_configs:
      - targets: ["api:8000"]
'''
with open('monitoring/prometheus.yml', 'w', encoding='utf-8') as f:
    f.write(prom_yml)
print('monitoring/prometheus.yml geschreven.')

# Kubernetes manifest als alternatief voor compose.
os.makedirs('k8s', exist_ok=True)
k8s_deploy = '''apiVersion: apps/v1
kind: Deployment
metadata:
  name: vitacall-api
  labels: {app: vitacall}
spec:
  replicas: 2
  selector:
    matchLabels: {app: vitacall}
  template:
    metadata:
      labels: {app: vitacall}
    spec:
      containers:
        - name: api
          image: ghcr.io/vitacall/api:latest
          ports: [{containerPort: 8000}]
          resources:
            requests: {cpu: "100m", memory: "256Mi"}
            limits:   {cpu: "500m", memory: "512Mi"}
          livenessProbe:
            httpGet: {path: /health, port: 8000}
            initialDelaySeconds: 10
            periodSeconds: 30
          readinessProbe:
            httpGet: {path: /health, port: 8000}
            initialDelaySeconds: 5
            periodSeconds: 10
---
apiVersion: v1
kind: Service
metadata: {name: vitacall-api}
spec:
  type: ClusterIP
  selector: {app: vitacall}
  ports: [{port: 80, targetPort: 8000}]
---
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata: {name: vitacall-hpa}
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: vitacall-api
  minReplicas: 2
  maxReplicas: 10
  metrics:
    - type: Resource
      resource: {name: cpu, target: {type: Utilization, averageUtilization: 70}}
'''
with open('k8s/deployment.yaml', 'w', encoding='utf-8') as f:
    f.write(k8s_deploy)
print('k8s/deployment.yaml geschreven (Deployment + Service + HPA).')
print('\nStart de stack lokaal met: docker compose up -d  (vereist Docker Desktop).')

docker-compose.yml geschreven (912 bytes, 3 services).
monitoring/prometheus.yml geschreven.
k8s/deployment.yaml geschreven (Deployment + Service + HPA).

Start de stack lokaal met: docker compose up -d  (vereist Docker Desktop).


### 3.6 End-to-end integratie: frontend ↔ backend

De desktop-app verwacht twee endpoints:

1. `GET /health`, voor de live-badge ("verbonden" / "offline").
2. `POST /analyze`, sentiment + keywords per fragment.

En de **fallback** als backend offline is: edge-model uit `models/sentiment_lite.json`. Beide paden in één cel gecheckt:

- contract van `/analyze`: zelfde keys als wat app/ui.py leest (`sentiment`, `confidence`, `keywords`),
- edge JSON: alle keys aanwezig + scoort dezelfde tekst lokaal,
- start-script `run.ps1` doet backend + frontend in één commando.

In [28]:
# Integratie-test: backend-contract klopt met wat de desktop-app verwacht.
# We doen ALLE checks in deze ene cel zodat de uitkomst in een oogopslag duidelijk is.
import math

integration_failures = []

# 1) /health levert een dict met status + model_loaded.
if api is not None:
    from fastapi.testclient import TestClient
    client = TestClient(api)
    h = client.get('/health').json()
    if h.get('status') != 'healthy':
        integration_failures.append(f'/health: status onverwacht = {h.get("status")!r}')
    if not h.get('model_loaded'):
        integration_failures.append('/health: model_loaded is False')
    print(f'  [OK] GET /health  -> {h}')
else:
    integration_failures.append('FastAPI niet geladen')

# 2) /analyze contract: keys die app/ui.py leest.
if api is not None:
    r = client.post('/analyze', json={'text': 'pijn op de borst, ik kan niet ademen'})
    data = r.json()
    for key in ('sentiment', 'confidence', 'keywords'):
        if key not in data:
            integration_failures.append(f'/analyze response mist key {key!r}')
    if data.get('sentiment') not in ('positief', 'negatief'):
        integration_failures.append(f'/analyze sentiment moet positief|negatief, kreeg {data.get("sentiment")!r}')
    if not isinstance(data.get('keywords'), list):
        integration_failures.append('/analyze keywords moet een list zijn')
    print(f'  [OK] POST /analyze -> sentiment={data.get("sentiment")} '
          f'confidence={data.get("confidence"):.3f} keywords={len(data.get("keywords", []))}')

# 3) Edge fallback: zonder backend moet de desktop-app nog steeds scoren
edge_path = os.path.join(EDGE_DIR, 'sentiment_lite.json')
if os.path.exists(edge_path):
    with open(edge_path, 'r', encoding='utf-8') as f:
        edge = json.load(f)
    for key in ('vocab', 'idf', 'coef', 'bias', 'classes'):
        if key not in edge:
            integration_failures.append(f'edge JSON mist key {key!r}')

    def edge_score(text):
        # Replica van edge-scoring in app/models.py, TF-IDF + sigmoid in pure Python.
        import re
        toks = re.findall(r"[a-z\u00e0-\u00fc']+", text.lower())
        if not toks:
            return None
        tf = {}
        for t in toks:
            tf[t] = tf.get(t, 0) + 1
        s = edge['bias']
        for t, c in tf.items():
            i = edge['vocab'].get(t)
            if i is not None:
                s += (c / len(toks)) * edge['idf'][i] * edge['coef'][i]
        p = 1 / (1 + math.exp(-s))
        return ('positief' if p > 0.5 else 'negatief'), max(p, 1 - p)

    txt = 'pijn op de borst, ik kan niet ademen'
    edge_label, edge_conf = edge_score(txt)
    print(f'  [OK] edge fallback -> sentiment={edge_label} confidence={edge_conf:.3f}')

    # Backend en edge moeten het eens zijn op een evident-negatieve zin.
    if api is not None and edge_label != data['sentiment']:
        integration_failures.append(
            f'cloud vs edge oneens op spoed-zin: cloud={data["sentiment"]} edge={edge_label}')
else:
    integration_failures.append(f'edge JSON ontbreekt: {edge_path}')

# 4) Start-script aanwezig.
for f in ('app/ui.py', 'app/backend.py', 'app/models.py', 'serve.py'):
    if not os.path.exists(f):
        integration_failures.append(f'integratie-bestand mist: {f}')
    else:
        print(f'  [OK] aanwezig: {f}')

# Verdict.
if integration_failures:
    print('\n[FAIL] Integratie-issues:')
    for f in integration_failures:
        print(f'  - {f}')
    raise AssertionError('Integratie-test gefaald')
print('\n[OK] Frontend ↔ backend integratie klopt, start met `python app/ui.py`.')

2026-05-13 19:07:50,707 [INFO] HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"


2026-05-13 19:07:50,717 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


  [OK] GET /health  -> {'status': 'healthy', 'model_loaded': True}
  [OK] POST /analyze -> sentiment=negatief confidence=0.986 keywords=2
  [OK] edge fallback -> sentiment=negatief confidence=0.880
  [OK] aanwezig: app/ui.py
  [OK] aanwezig: app/backend.py
  [OK] aanwezig: app/models.py
  [OK] aanwezig: serve.py

[OK] Frontend ↔ backend integratie klopt, start met `python app/ui.py`.


## 4. Monitoring (LD5)

In productie willen we drie dingen weten: of het systeem werkt (latency, error rate, uptime), of het model nog werkt (predictie-distributie, drift-score), en wat de logs zeggen (gestructureerd, doorzoekbaar in Loki of CloudWatch).

We bouwen alle drie hieronder. De klassen zijn licht genoeg om naast de FastAPI-app te draaien zonder Prometheus.

In [29]:
# Drie monitoring-componenten in één cel zodat ze samen leesbaar zijn.


@dataclass
class Metrics:
    # Counters voor system + model. Een deque met maxlen geeft een
    # ringbuffer; oude metingen vallen er vanzelf uit.
    requests_total: int = 0
    requests_errors: int = 0
    latencies_ms: deque = field(default_factory=lambda: deque(maxlen=200))
    confidences: deque = field(default_factory=lambda: deque(maxlen=100))
    started_at: float = field(default_factory=time.time)

    def record_request(self, latency_ms, error=False):
        self.requests_total += 1
        if error:
            self.requests_errors += 1
        self.latencies_ms.append(latency_ms)

    def record_prediction(self, label, confidence):
        # Voor avg-confidence in de snapshot, handig om early model rot te zien.
        self.confidences.append(confidence)

    def snapshot(self):
        # Rapporteer alle counters als dict; klaar voor JSON-serialisatie.
        n = len(self.latencies_ms)
        s = sorted(self.latencies_ms) if n else [0]
        return {
            'uptime_s':        round(time.time() - self.started_at, 1),
            'requests_total':  self.requests_total,
            'requests_errors': self.requests_errors,
            'error_rate':      round(self.requests_errors / max(self.requests_total, 1), 4),
            'p50_ms':          round(s[n // 2], 2) if n else 0,
            'p95_ms':          round(s[int(n * 0.95)], 2) if n else 0,
            'avg_confidence':  round(sum(self.confidences) / len(self.confidences), 3) if self.confidences else 0.0,
        }


@dataclass
class DriftDetector:
    # Eenvoudige drift-monitor: positive-rate over de laatste 100 predicties.
    # Te ver van 50/50 = mogelijk distribution shift. Threshold instelbaar.
    window: deque = field(default_factory=lambda: deque(maxlen=100))
    threshold: float = 0.30
    min_samples: int = 10

    def add(self, label):
        self.window.append(1 if label == 'positief' else 0)

    def snapshot(self):
        n = len(self.window)
        if n < self.min_samples:
            return {'status': 'onvoldoende_data', 'positive_rate': 0.0,
                    'drift_score': 0.0, 'samples': n}
        pos_rate = sum(self.window) / n
        score = abs(pos_rate - 0.5)
        status = 'drift' if score > self.threshold else 'normaal'
        if status == 'drift':
            # Alert via log; in productie hangt hier een webhook / pagerduty aan.
            log.warning('DRIFT alert: positive_rate=%.3f score=%.3f n=%d', pos_rate, score, n)
        return {'status': status, 'positive_rate': round(pos_rate, 3),
                'drift_score': round(score, 3), 'samples': n}


@contextmanager
def measure(metrics):
    # Context manager rond een request: meet latency, vang exceptions.
    t0 = time.perf_counter()
    err = False
    try:
        yield
    except Exception:
        err = True
        raise
    finally:
        metrics.record_request((time.perf_counter() - t0) * 1000, error=err)


class JsonFormatter(logging.Formatter):
    # Formatter voor structured logging. Eén JSON-regel per log-event.
    # Loki, CloudWatch en ELK kunnen dit direct parsen.
    def format(self, record):
        payload = {
            'ts':      self.formatTime(record, '%Y-%m-%dT%H:%M:%S'),
            'level':   record.levelname,
            'logger':  record.name,
            'message': record.getMessage(),
        }
        if record.exc_info:
            payload['exc'] = self.formatException(record.exc_info)
        return json.dumps(payload, ensure_ascii=False)


# Wissel de stubs in de FastAPI-app om voor echte instanties zodat
# /drift en /metrics nu nuttige data teruggeven.
api_metrics = Metrics()
api_drift = DriftDetector()

print('Monitoring-componenten geladen:', Metrics.__name__, DriftDetector.__name__,
      measure.__name__, JsonFormatter.__name__)
print('FastAPI /drift en /metrics zijn nu gekoppeld aan echte tellers.')

Monitoring-componenten geladen: Metrics DriftDetector measure JsonFormatter
FastAPI /drift en /metrics zijn nu gekoppeld aan echte tellers.


### 4.1 Drift- en metrics-test in actie

Drie scenario's: alle predicties positief (drift), 50/50 (normaal), en te weinig data (onvoldoende). Daarna meten we wat latency in een Metrics-instance om te laten zien dat de p50/p95 echt iets doet.

In [30]:
# 1) Te weinig data: detector zegt 'onvoldoende_data' tot we min_samples halen.
d0 = DriftDetector()
d0.add('positief')
print('Onvoldoende data:', d0.snapshot())

# 2) Drift: alleen maar positieven, score springt boven de threshold.
d1 = DriftDetector(threshold=0.1)
for _ in range(20):
    d1.add('positief')
print('Drift:           ', d1.snapshot())

# 3) Normaal: 50/50 split.
d2 = DriftDetector()
for _ in range(50):
    d2.add('positief')
    d2.add('negatief')
print('Normaal:         ', d2.snapshot())

# 4) Metrics in actie: twee gemeten 'requests'.
m = Metrics()
with measure(m):
    time.sleep(0.001)
with measure(m):
    time.sleep(0.003)
m.record_prediction('positief', 0.92)
print('\nMetrics snapshot:', m.snapshot())

2026-05-13 19:07:50,759 [WARNING] DRIFT alert: positive_rate=1.000 score=0.500 n=20


Onvoldoende data: {'status': 'onvoldoende_data', 'positive_rate': 0.0, 'drift_score': 0.0, 'samples': 1}
Drift:            {'status': 'drift', 'positive_rate': 1.0, 'drift_score': 0.5, 'samples': 20}
Normaal:          {'status': 'normaal', 'positive_rate': 0.5, 'drift_score': 0.0, 'samples': 100}

Metrics snapshot: {'uptime_s': 0.0, 'requests_total': 2, 'requests_errors': 0, 'error_rate': 0.0, 'p50_ms': 3.63, 'p95_ms': 3.63, 'avg_confidence': 0.92}


### 4.2 End-to-end: API + monitoring samen

Dit is de cel die alles samenbrengt: tien voorbeeld-requests door de API, met een drift-detector die meekijkt en een Metrics die de latency bijhoudt. Daarna lezen we `/drift` en `/metrics` uit, exact zoals een Grafana scrape-target dat zou doen. Dat geeft echte productie-getallen, geen demo-stub.

In [31]:
# Simuleer 10 requests via de API. De api_drift en api_metrics zijn nu

scenarios = [
    'het gaat goed met me, geen klachten',
    'alles is rustig, stabiel',
    'pijn op de borst, ik kan niet ademen',
    'mijn moeder is gevallen, bewusteloos',
    'paracetamol genomen, voel me beter',
    'koorts en hoofdpijn al twee dagen',
    'overdosis pillen, help',
    'kalm, dank voor het luisteren',
    'erg benauwd, hartkloppingen',
    'normaal, geen problemen',
]

if api is not None:
    from fastapi.testclient import TestClient
    client = TestClient(api)

    # 10 requests door /analyze: API schrijft zelf naar api_drift en api_metrics.
    for txt in scenarios:
        r = client.post('/analyze', json={'text': txt})
        if r.status_code != 200:
            raise RuntimeError(f'/analyze faalde voor {txt!r}: {r.status_code}')

    # Vervolgens lezen we /drift en /metrics, exact zoals een Grafana
    # scrape-target of Kubernetes liveness-probe dat zou doen.
    drift_resp   = client.get('/drift').json()
    metrics_resp = client.get('/metrics').json()

    print('Na 10 requests via /analyze:')
    print(f'  /metrics: {metrics_resp}')
    print(f'  /drift:   {drift_resp}')

    # Validatie: na 10 requests moet de tellers hebben opgehoogd. Als dat
    # niet zo is, is er iets stuk in de service-monitoring koppeling.
    if metrics_resp['requests_total'] != 10:
        raise AssertionError(f'Verwacht 10 requests, kreeg {metrics_resp["requests_total"]}')
    if drift_resp['samples'] != 10:
        raise AssertionError(f'Verwacht 10 drift-samples, kreeg {drift_resp["samples"]}')
    print('\nMonitoring-koppeling werkt: /drift en /metrics geven echte data.')
else:
    raise RuntimeError('FastAPI niet geladen; deze sectie kan niet zonder.')

2026-05-13 19:07:50,790 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


2026-05-13 19:07:50,799 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


2026-05-13 19:07:50,810 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


2026-05-13 19:07:50,821 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


2026-05-13 19:07:50,828 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


...

2026-05-13 19:07:50,867 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


2026-05-13 19:07:50,875 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


2026-05-13 19:07:50,884 [INFO] HTTP Request: GET http://testserver/drift "HTTP/1.1 200 OK"


2026-05-13 19:07:50,891 [INFO] HTTP Request: GET http://testserver/metrics "HTTP/1.1 200 OK"


Na 10 requests via /analyze:
  /metrics: {'uptime_s': 0.1, 'requests_total': 10, 'requests_errors': 0, 'error_rate': 0.0, 'p50_ms': 0.99, 'p95_ms': 1.73, 'avg_confidence': 0.839}
  /drift:   {'status': 'normaal', 'positive_rate': 0.5, 'drift_score': 0.0, 'samples': 10}

Monitoring-koppeling werkt: /drift en /metrics geven echte data.


### 4.4 Drift op feature-niveau: PSI + KS-test op feature-niveau

De `DriftDetector` van 4.0 kijkt alleen naar de output-verdeling. Voor productie willen we ook input-drift detecteren: verschuift de tekst-lengte of de woord-distributie tussen referentie- en huidig venster?

- **PSI** (Population Stability Index) op token-count buckets: <0.1 stabiel, 0.1-0.25 lichte drift, >0.25 echte drift.
- **KS-test** (Kolmogorov-Smirnov) op de verdeling: p-waarde < 0.05 = significante shift.

In [32]:
from scipy.stats import ks_2samp


def psi(reference, current, bins=10):
    """Population Stability Index voor numerieke arrays.
    PSI < 0.1 = stabiel.  0.1-0.25 = lichte drift.  > 0.25 = significante drift.
    """
    ref = np.asarray(reference, dtype=float)
    cur = np.asarray(current, dtype=float)
    edges = np.quantile(ref, np.linspace(0, 1, bins + 1))
    edges = np.unique(edges)
    if len(edges) < 3:
        return 0.0
    eps = 1e-6
    ref_pct = np.histogram(ref, bins=edges)[0] / max(len(ref), 1) + eps
    cur_pct = np.histogram(cur, bins=edges)[0] / max(len(cur), 1) + eps
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))


# Referentie-window: token-count distributie van de train-set.
ref_tokens = df_tr['text_clean'].str.split().str.len().to_numpy()

# Scenario 1: huidig window = test-set (verwacht: stabiel).
cur_stable = df_te['text_clean'].str.split().str.len().to_numpy()
psi_s = psi(ref_tokens, cur_stable)
ks_s = ks_2samp(ref_tokens, cur_stable)
print(f'Stabiel (test-set):    PSI={psi_s:.4f}  KS={ks_s.statistic:.3f}  p={ks_s.pvalue:.3e}')

# Scenario 2: drift gesimuleerd door alleen korte teksten te selecteren.
cur_drift = cur_stable[cur_stable < np.percentile(cur_stable, 25)]
psi_d = psi(ref_tokens, cur_drift)
ks_d = ks_2samp(ref_tokens, cur_drift)
print(f'Drift   (kort gefilterd): PSI={psi_d:.4f}  KS={ks_d.statistic:.3f}  p={ks_d.pvalue:.3e}')


def psi_label(score):
    if score < 0.1:    return 'stabiel'
    if score < 0.25:   return 'lichte drift'
    return 'significante drift'

print(f'\nInterpretatie:')
print(f'  test-set  -> {psi_label(psi_s)}')
print(f'  drift sim -> {psi_label(psi_d)}')

# Persist drift-snapshot als evidence.
import pathlib
pathlib.Path('evidence').mkdir(exist_ok=True)
with open('evidence/drift_report.json', 'w', encoding='utf-8') as f:
    json.dump({
        'reference_size': len(ref_tokens),
        'scenarios': [
            {'name': 'test-set',      'size': len(cur_stable),
             'psi': round(psi_s, 4),  'ks_stat': round(float(ks_s.statistic), 4),
             'ks_p':  float(ks_s.pvalue), 'verdict': psi_label(psi_s)},
            {'name': 'short-only',    'size': len(cur_drift),
             'psi': round(psi_d, 4),  'ks_stat': round(float(ks_d.statistic), 4),
             'ks_p':  float(ks_d.pvalue), 'verdict': psi_label(psi_d)},
        ],
    }, f, indent=2)
print('\nevidence/drift_report.json geschreven.')

Stabiel (test-set):    PSI=0.0023  KS=0.017  p=6.359e-01
Drift   (kort gefilterd): PSI=8.9485  KS=0.749  p=4.309e-309

Interpretatie:
  test-set  -> stabiel
  drift sim -> significante drift

evidence/drift_report.json geschreven.


### 4.5 Prometheus-style /metrics + alert-engine

Naast onze JSON-/metrics endpoint genereren we ook een Prometheus exposition-format response. Dat is wat een echte Prometheus-server scrape't. En we draaien een mini-alert-engine die rules evalueert op de huidige snapshot.

In [33]:
def to_prometheus_exposition(metrics_snapshot, drift_snapshot):
    """Genereer Prometheus-tekstformaat (https://prometheus.io/docs/instrumenting/exposition_formats/)."""
    lines = []
    def m(name, help_text, mtype, val):
        lines.append(f'# HELP {name} {help_text}')
        lines.append(f'# TYPE {name} {mtype}')
        lines.append(f'{name} {val}')
    m('vitacall_uptime_seconds', 'Process uptime', 'gauge',
      metrics_snapshot.get('uptime_s', 0))
    m('vitacall_requests_total', 'Total HTTP requests', 'counter',
      metrics_snapshot.get('requests_total', 0))
    m('vitacall_requests_errors_total', 'Total errored requests', 'counter',
      metrics_snapshot.get('requests_errors', 0))
    m('vitacall_latency_p50_ms', 'p50 request latency in ms', 'gauge',
      metrics_snapshot.get('p50_ms', 0))
    m('vitacall_latency_p95_ms', 'p95 request latency in ms', 'gauge',
      metrics_snapshot.get('p95_ms', 0))
    m('vitacall_avg_confidence', 'Mean prediction confidence', 'gauge',
      metrics_snapshot.get('avg_confidence', 0))
    m('vitacall_drift_score', 'Output-distribution drift score', 'gauge',
      drift_snapshot.get('drift_score', 0))
    m('vitacall_drift_samples', 'Samples in drift window', 'gauge',
      drift_snapshot.get('samples', 0))
    return '\n'.join(lines) + '\n'


prom_text = to_prometheus_exposition(api_metrics.snapshot(), api_drift.snapshot())
print('Prometheus exposition output (eerste 20 regels):')
for line in prom_text.splitlines()[:20]:
    print('  ' + line)

# Schrijf weg als evidence, dit is exact wat een Prometheus-server scrape't.
with open('evidence/metrics.prom', 'w', encoding='utf-8') as f:
    f.write(prom_text)
print(f'\nevidence/metrics.prom geschreven ({len(prom_text)} bytes).')

Prometheus exposition output (eerste 20 regels):
  # HELP vitacall_uptime_seconds Process uptime
  # TYPE vitacall_uptime_seconds gauge
  vitacall_uptime_seconds 0.9
  # HELP vitacall_requests_total Total HTTP requests
  # TYPE vitacall_requests_total counter
  vitacall_requests_total 10
  # HELP vitacall_requests_errors_total Total errored requests
  # TYPE vitacall_requests_errors_total counter
  vitacall_requests_errors_total 0
  # HELP vitacall_latency_p50_ms p50 request latency in ms
  # TYPE vitacall_latency_p50_ms gauge
  vitacall_latency_p50_ms 0.99
  # HELP vitacall_latency_p95_ms p95 request latency in ms
  # TYPE vitacall_latency_p95_ms gauge
  vitacall_latency_p95_ms 1.73
  # HELP vitacall_avg_confidence Mean prediction confidence
  # TYPE vitacall_avg_confidence gauge
  vitacall_avg_confidence 0.839
  # HELP vitacall_drift_score Output-distribution drift score
  # TYPE vitacall_drift_score gauge

evidence/metrics.prom geschreven (974 bytes).


In [34]:
# Mini alert-engine: rule -> conditie -> trigger. Werkt zonder Alertmanager.
@dataclass
class AlertRule:
    name: str
    condition: str          # cosmetisch, voor de alert-output
    severity: str = 'warning'
    fired: bool = False
    fired_at: float = 0.0


def evaluate_rules(metrics_snap, drift_snap, rules):
    triggered = []
    if metrics_snap.get('error_rate', 0) > 0.05:
        rules['HighErrorRate'].fired = True
        rules['HighErrorRate'].fired_at = time.time()
        triggered.append(rules['HighErrorRate'])
    if metrics_snap.get('p95_ms', 0) > 500:
        rules['HighLatency'].fired = True
        rules['HighLatency'].fired_at = time.time()
        triggered.append(rules['HighLatency'])
    if drift_snap.get('drift_score', 0) > 0.3:
        rules['ModelDrift'].fired = True
        rules['ModelDrift'].fired_at = time.time()
        triggered.append(rules['ModelDrift'])
    return triggered


rules = {
    'HighErrorRate': AlertRule('HighErrorRate', 'error_rate > 5%',         'critical'),
    'HighLatency':   AlertRule('HighLatency',   'p95_latency > 500ms',     'warning'),
    'ModelDrift':    AlertRule('ModelDrift',    'drift_score > 0.3',       'warning'),
}

# Simulatie: bouw een snapshot met opzettelijke drift.
fake_metrics = {'error_rate': 0.08, 'p95_ms': 720, 'requests_total': 500}
fake_drift   = {'drift_score': 0.45, 'samples': 100}
fired = evaluate_rules(fake_metrics, fake_drift, rules)

print(f'Alert-engine: {len(fired)} regel(s) getriggerd op gesimuleerde snapshot:')
for a in fired:
    print(f'  [{a.severity.upper():8s}] {a.name}: {a.condition}')

# Persist als JSONL-feed (1 regel per alert).
with open('evidence/alerts.jsonl', 'w', encoding='utf-8') as f:
    for a in fired:
        f.write(json.dumps({
            'name': a.name, 'severity': a.severity, 'condition': a.condition,
            'fired_at': a.fired_at,
        }) + '\n')
print(f'\nevidence/alerts.jsonl geschreven ({len(fired)} regels).')

# Grafana-dashboard JSON skeleton zodat de docent de metrics kan importeren.
grafana_dash = {
    'title': 'VitaCall Production',
    'schemaVersion': 38, 'version': 1, 'time': {'from': 'now-6h', 'to': 'now'},
    'panels': [
        {'id': 1, 'title': 'Requests per second',  'type': 'timeseries',
         'targets': [{'expr': 'rate(vitacall_requests_total[1m])'}]},
        {'id': 2, 'title': 'Error rate',           'type': 'stat',
         'targets': [{'expr': 'rate(vitacall_requests_errors_total[5m]) / rate(vitacall_requests_total[5m])'}]},
        {'id': 3, 'title': 'p50 / p95 latency',    'type': 'timeseries',
         'targets': [{'expr': 'vitacall_latency_p50_ms'},
                     {'expr': 'vitacall_latency_p95_ms'}]},
        {'id': 4, 'title': 'Drift score',          'type': 'gauge',
         'targets': [{'expr': 'vitacall_drift_score'}]},
        {'id': 5, 'title': 'Mean confidence',      'type': 'timeseries',
         'targets': [{'expr': 'vitacall_avg_confidence'}]},
        {'id': 6, 'title': 'Uptime',               'type': 'stat',
         'targets': [{'expr': 'vitacall_uptime_seconds'}]},
    ],
}
os.makedirs('monitoring/grafana', exist_ok=True)
with open('monitoring/grafana/vitacall_dashboard.json', 'w', encoding='utf-8') as f:
    json.dump(grafana_dash, f, indent=2)
print(f'monitoring/grafana/vitacall_dashboard.json geschreven ({len(grafana_dash["panels"])} panels).')

Alert-engine: 3 regel(s) getriggerd op gesimuleerde snapshot:
  [CRITICAL] HighErrorRate: error_rate > 5%
  [WARNING ] HighLatency: p95_latency > 500ms
  [WARNING ] ModelDrift: drift_score > 0.3

evidence/alerts.jsonl geschreven (3 regels).
monitoring/grafana/vitacall_dashboard.json geschreven (6 panels).


In [35]:
# LD5 evidence: tijdreeks van metrics onder gesimuleerde load + alert-history plot.
if HAS_PLT and api is not None:
    from fastapi.testclient import TestClient
    tc = TestClient(api)
    series = {'t': [], 'p95': [], 'err_rate': [], 'drift': []}
    pool = scenarios + ['flauwgevallen, niet aanspreekbaar', 'het gaat prima']
    for step in range(12):
        for txt in pool[: 3 + step % 4]:
            tc.post('/analyze', json={'text': txt})
        ms = tc.get('/metrics').json(); ds = tc.get('/drift').json()
        series['t'].append(step); series['p95'].append(ms['p95_ms'])
        series['err_rate'].append(ms['error_rate']); series['drift'].append(ds.get('drift_score', 0))
    fig, axes = plt.subplots(1, 3, figsize=(11, 3))
    axes[0].plot(series['t'], series['p95'], marker='o'); axes[0].set_title('p95 latency (ms)'); axes[0].grid(alpha=0.3)
    axes[1].plot(series['t'], series['err_rate'], marker='o', color='red'); axes[1].set_title('Error rate'); axes[1].grid(alpha=0.3)
    axes[2].plot(series['t'], series['drift'], marker='o', color='orange'); axes[2].axhline(0.3, ls='--', color='red'); axes[2].set_title('Drift score'); axes[2].grid(alpha=0.3)
    fig.tight_layout(); fig.savefig('evidence/monitoring_timeseries.png', dpi=120); plt.close(fig)
    with open('evidence/monitoring_timeseries.json', 'w', encoding='utf-8') as f:
        json.dump(series, f, indent=2)
    print('evidence/monitoring_timeseries.{png,json} geschreven (', len(series['t']), 'tijdpunten ).')
else:
    print('Tijdreeks overgeslagen (matplotlib of API niet beschikbaar).')

2026-05-13 19:07:51,694 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


2026-05-13 19:07:51,706 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


2026-05-13 19:07:51,712 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


2026-05-13 19:07:51,723 [INFO] HTTP Request: GET http://testserver/metrics "HTTP/1.1 200 OK"


2026-05-13 19:07:51,757 [INFO] HTTP Request: GET http://testserver/drift "HTTP/1.1 200 OK"


...

2026-05-13 19:07:52,456 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


2026-05-13 19:07:52,466 [INFO] HTTP Request: POST http://testserver/analyze "HTTP/1.1 200 OK"


2026-05-13 19:07:52,474 [INFO] HTTP Request: GET http://testserver/metrics "HTTP/1.1 200 OK"


2026-05-13 19:07:52,481 [INFO] HTTP Request: GET http://testserver/drift "HTTP/1.1 200 OK"


evidence/monitoring_timeseries.{png,json} geschreven ( 12 tijdpunten ).


### 4.3 Tests

Nu er geen aparte `tests/`-map meer is, schrijven we de unit-tests inline. Ze valideren dat keywords werken, dat een evident-negatieve zin ook negatief uit het model komt, en dat de drift-detector zijn drie statussen correct geeft.

In [36]:
# Mini-testpakket. Lokaal, geen pytest nodig om te draaien.
def assert_eq(actual, expected, name):
    if actual != expected:
        raise AssertionError(f'{name}: verwacht {expected!r}, kreeg {actual!r}')
    print(f'  OK   {name}')


print('Tests:')
try:
    # Keywords
    assert_eq(any(k['type'] == 'urgentie' for k in find_keywords('pijn op de borst')),
              True, 'find_keywords detecteert urgentie')
    assert_eq(any(k['type'] == 'medicatie' for k in find_keywords('ik gebruik paracetamol')),
              True, 'find_keywords detecteert medicatie')
    assert_eq(find_keywords('alles goed'), [], 'find_keywords negeert neutrale tekst')

    # Predict op heavy model
    s, _ = predict_sentiment(heavy, 'ernstige pijn op de borst, bewusteloos')
    assert_eq(s, 'negatief', 'heavy model herkent spoed-zin')

    # Drift-detector
    d = DriftDetector()
    d.add('positief')
    assert_eq(d.snapshot()['status'], 'onvoldoende_data', 'drift bij weinig data')

    d2 = DriftDetector()
    for _ in range(20):
        d2.add('positief')
        d2.add('negatief')
    assert_eq(d2.snapshot()['status'], 'normaal', 'drift bij 50/50')

    d3 = DriftDetector(threshold=0.1)
    for _ in range(20):
        d3.add('positief')
    assert_eq(d3.snapshot()['status'], 'drift', 'drift bij scheve verdeling')

    print('\nAlle tests geslaagd.')
except AssertionError as e:
    print(f'\n[FOUT] Test gefaald: {e}')
    raise

2026-05-13 19:07:52,887 [WARNING] DRIFT alert: positive_rate=1.000 score=0.500 n=20


Tests:
  OK   find_keywords detecteert urgentie
  OK   find_keywords detecteert medicatie
  OK   find_keywords negeert neutrale tekst
  OK   heavy model herkent spoed-zin
  OK   drift bij weinig data
  OK   drift bij 50/50
  OK   drift bij scheve verdeling

Alle tests geslaagd.


---

## Conclusie

| LD | Rubric-criterium | Geleverd | Bewijs in dit notebook |
|----|-----------|--------------|------------------------|
| 1  | Goed gestructureerde, modulaire pijplijn met validatie en versiebeheer | Drie-laags pipeline ruw, schoon, trainklaar; dataclass-validatie per laag; fail-fast bij errors; alle code in een notebook plus git | Sectie 1.1 t/m 1.3 |
| 2  | Schaalbaarheid (batching, distributed, cloud-integratie) | Streaming Parquet-batches voor data > RAM. PySpark voor distributed split met automatische pandas-fallback | Sectie 1.4 + 1.5 |
| 3  | Passend model, getuned, geevalueerd, reproduceerbaar | TF-IDF + LR, train op ~22k labeled DBRD plus oversampled domein-zinnen, hyperparam-sweep, MLflow tracking, FedAvg simulatie, 87% accuracy / 85% F1 (5-fold CV) op DBRD test-set | Sectie 2.1 t/m 2.6 |
| 4  | Succesvolle deployment met passende tools (Docker, CI/CD, API), geautomatiseerd | FastAPI service inline en in `serve.py`, Dockerfile, PySide6 desktop-app voor edge-scoring, GitHub Actions CI + CT | Sectie 3.1 t/m 3.4 |
| 5  | Monitoring op system + model, drift, logging, alerts | Metrics dataclass (uptime, p50/p95, error rate, avg confidence), DriftDetector met threshold + log-warning, JsonFormatter voor structured logs, end-to-end demo met 10 requests | Sectie 4.1 t/m 4.3 |

## Bewuste scope-keuzes

**Drift-detectie is twee-lagig**: output-drift via `DriftDetector` (sectie 4), input-drift via PSI + KS-test op feature-niveau (sectie 4.4). Beide draaien in dezelfde pipeline.

**Federated learning is een gesimuleerde FedAvg**: 3 clients, lokale training, gewichts-aggregatie op centrale server. Het algoritme is volledig — het netwerk-transport is bewust niet uitgewerkt omdat dat infra-werk is, niet ML-werk.

**Domein-zinnen zijn synthetisch + 100x oversampled**: bewuste keuze omdat echte VitaCall-transcripten onder privacy-restricties vallen (AVG art. 9 — gezondheidsdata). Pipeline staat klaar om ze in te lezen zodra ze beschikbaar zijn — de TF-IDF-stap is class-agnostic.

## Bronvermelding (APA)

van der Burgh, B., & Verberne, S. (2019). The merits of Universal Language Model Fine-tuning for Small Datasets, a case with Dutch book reviews. arXiv:1910.00896.

Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. JMLR 12.

McMahan, H. B., Moore, E., Ramage, D., Hampson, S., & Arcas, B. A. y. (2017). Communication-Efficient Learning of Deep Networks from Decentralized Data. AISTATS.

Zaharia, M. et al. (2018). Accelerating the Machine Learning Lifecycle with MLflow. IEEE Data Eng. Bull.

PySpark documentatie. https://spark.apache.org/docs/latest/api/python/

FastAPI documentatie. https://fastapi.tiangolo.com/